# LegalRAG
## Hybrid Retrieval RAG and Evaluation Platform for Indian Court Judgments

This notebook is the main experimental pipeline for LegalRAG.

The structure follows the original project flow, now configured for the final **100,000 unique High Court judgments** experiment.

The project is evaluated experimentally rather than by simply claiming that one retrieval method is best:

**BM25 → Dense → Hybrid RRF → Cross-Encoder Reranker → RAG generation → Grounding / hallucination evaluation**

The experiment is designed for Kaggle and uses OmniRoute for all external model generation.


## 00 Configuration

In [7]:
from pathlib import Path
import os
import json
import re
import gc
import time
import random
import hashlib
import shutil

BASE_DIR = Path('/kaggle/working/legalrag_100k')
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Dataset
DATASET_NAME = 'overthelex/indian-court-decisions'
DATASET_CONFIG = 'high_courts'
DATASET_SPLIT = 'train'
TARGET_JUDGMENTS = 100_000
COLLECTION_SHARD_SIZE = 5_000
STREAM_SHUFFLE_BUFFER = 50_000
RANDOM_SEED = 42

# Cleaning

# Chunking
CHUNK_SIZE = 1_200
CHUNK_OVERLAP = 200
MIN_CHUNK_LENGTH = 100

# Evaluation benchmark
EVAL_DOCUMENTS = 500
PILOT_DOCUMENTS = 10
EVAL_DOCUMENT_MAX_CHARS = 80_000

# Question-type allocation for the final benchmark
QUESTION_TYPE_TARGETS = {
    'reasoning': 150,
    'outcome': 120,
    'fact': 100,
    'legal_provision': 80,
    'multi_hop': 50,
}

# Dense retrieval
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
EMBED_DIM = 768
EMBED_SHARD_SIZE = 5_000
EMBED_BATCH_SIZE = 32
QUERY_INSTRUCTION = 'Represent this sentence for searching relevant passages: '

# Retrieval candidate depth
RETRIEVAL_TOP_K = 50
RRF_K = 60
RERANK_CANDIDATES = 50
CONTEXT_TOP_K = 5

# Cross-encoder reranker
RERANKER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

# OmniRoute
OMNIROUTE_BASE_URL = 'https://mate-statutory-temp-planners.trycloudflare.com/v1'
OMNIROUTE_MODEL = 'kaggle'
OMNIROUTE_SECRET_NAME = 'OMNIROUTE_API_KEY'

# Paths
MANIFEST_PATH = BASE_DIR / 'project_manifest.json'
COLLECTION_MANIFEST_PATH = BASE_DIR / 'collection_manifest.json'
CLEAN_PATH = BASE_DIR / 'legal_judgments_clean.parquet'
CHUNKS_PATH = BASE_DIR / 'legal_chunks.parquet'
EVAL_DOC_PATH = BASE_DIR / 'evaluation_documents.parquet'
PILOT_PATH = BASE_DIR / 'eval_pilot_10.json'
CANDIDATES_PATH = BASE_DIR / 'eval_candidates.json'
GOLD_EVAL_PATH = BASE_DIR / 'gold_eval.json'

CURRENT_MANIFEST = {
    'dataset': DATASET_NAME,
    'config': DATASET_CONFIG,
    'split': DATASET_SPLIT,
    'target_judgments': TARGET_JUDGMENTS,
    'random_seed': RANDOM_SEED,
    'chunk_size': CHUNK_SIZE,
    'chunk_overlap': CHUNK_OVERLAP,
    'min_chunk_length': MIN_CHUNK_LENGTH,
    'eval_documents': EVAL_DOCUMENTS,
    'embedding_model': MODEL_NAME,
    'embedding_dimension': EMBED_DIM,
    'retrieval_top_k': RETRIEVAL_TOP_K,
    'rrf_k': RRF_K,
    'reranker_model': RERANKER_MODEL,
    'reranker_candidates': RERANK_CANDIDATES,
    'context_top_k': CONTEXT_TOP_K,
    'omniroute_model': OMNIROUTE_MODEL,
}

if MANIFEST_PATH.exists():
    previous_manifest = json.loads(MANIFEST_PATH.read_text())
    if previous_manifest != CURRENT_MANIFEST:
        raise RuntimeError(
            'BASE_DIR contains a different LegalRAG experiment. '
            'Use a fresh /kaggle/working/legalrag_100k directory rather than mixing runs.'
        )
else:
    MANIFEST_PATH.write_text(
        json.dumps(CURRENT_MANIFEST, indent=2)
    )

print('Project directory:', BASE_DIR)
print('Target judgments:', f'{TARGET_JUDGMENTS:,}')
print('Evaluation benchmark:', f'{EVAL_DOCUMENTS:,} questions')
print('Dense model:', MODEL_NAME)
print('Hybrid candidate depth:', RETRIEVAL_TOP_K)
print('Reranker candidates:', RERANK_CANDIDATES)
print('RAG context:', CONTEXT_TOP_K)
print('OmniRoute model:', OMNIROUTE_MODEL)


RuntimeError: BASE_DIR contains a different LegalRAG experiment. Use a fresh /kaggle/working/legalrag_100k directory rather than mixing runs.

## 01 Environment validation

In [6]:
import sys
import psutil
import torch

print('Python:', sys.version.split()[0])
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB')
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f'GPU {i}: {torch.cuda.get_device_name(i)} | '
        f'VRAM {props.total_memory / 1024**3:.1f} GB'
    )

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for the dense embedding stage.')

if torch.cuda.device_count() < 2:
    raise RuntimeError('This notebook expects two CUDA GPUs for dense embeddings.')

Python: 3.12.13
RAM: 31.3 GB
CUDA available: True
GPU count: 2
GPU 0: Tesla T4 | VRAM 14.6 GB
GPU 1: Tesla T4 | VRAM 14.6 GB


## 02 Install the small set of dependencies

In [7]:
# Avoid broad environment upgrades. Install only what this notebook uses.
!pip -q install datasets pyarrow tqdm langchain-text-splitters rank-bm25 transformers sentencepiece faiss-cpu openai sentence-transformers


## 03 Dataset collection

In [9]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

collection_files = sorted(BASE_DIR.glob('judgments_*.parquet'))
existing_rows = 0
seen_cnrs = set()
seen_hashes = set()

if collection_files:
    print('Existing collection shards found. Checking progress...')
    for path in collection_files:
        part = pd.read_parquet(path, columns=['cnr', 'full_text'])
        existing_rows += len(part)
        seen_cnrs.update(part['cnr'].fillna('').astype(str))
        for text in part['full_text'].fillna('').astype(str):
            seen_hashes.add(
                hashlib.sha1(
                    text.encode('utf-8', errors='ignore')
                ).hexdigest()
            )
        del part

if existing_rows == TARGET_JUDGMENTS:
    print(
        f'Existing complete collection found: {existing_rows:,} judgments — skipping collection.'
    )
else:
    print(
        f'Collection progress: {existing_rows:,}/{TARGET_JUDGMENTS:,}'
    )

    ds = load_dataset(
        DATASET_NAME,
        DATASET_CONFIG,
        split=DATASET_SPLIT,
        streaming=True,
    )

    ds = ds.shuffle(
        seed=RANDOM_SEED,
        buffer_size=STREAM_SHUFFLE_BUFFER,
    )

    buffer = []
    total_saved = existing_rows
    shard_id = max(
        [int(path.stem.split('_')[-1]) for path in collection_files],
        default=-1
    ) + 1

    fields = [
        'cnr', 'source', 'court_code', 'court_name', 'bench', 'year',
        'full_text', 'text_length', 'title', 'judge', 'petitioner',
        'respondent', 'decision_date', 'disposal_nature',
        'disposal_nature_normalized', 'case_type'
    ]

    with tqdm(
        total=TARGET_JUDGMENTS,
        initial=existing_rows,
        desc='Collecting judgments'
    ) as pbar:

        for row in ds:

            if total_saved >= TARGET_JUDGMENTS:
                break

            cnr = str(row.get('cnr') or '').strip()
            text = str(row.get('full_text') or '').strip()

            if not cnr or not text:
                continue

            if cnr in seen_cnrs:
                continue

            text_hash = hashlib.sha1(
                text.encode('utf-8', errors='ignore')
            ).hexdigest()

            if text_hash in seen_hashes:
                continue

            seen_cnrs.add(cnr)
            seen_hashes.add(text_hash)

            buffer.append({
                field: row.get(field)
                for field in fields
            })

            total_saved += 1
            pbar.update(1)

            if len(buffer) >= COLLECTION_SHARD_SIZE:
                path = BASE_DIR / f'judgments_{shard_id:03d}.parquet'
                pd.DataFrame(buffer).to_parquet(
                    path,
                    index=False
                )
                print(f'\nSaved: {path.name}')
                buffer.clear()
                shard_id += 1
                gc.collect()

    if buffer:
        path = BASE_DIR / f'judgments_{shard_id:03d}.parquet'
        pd.DataFrame(buffer).to_parquet(
            path,
            index=False
        )
        print(f'\nSaved: {path.name}')

    if total_saved != TARGET_JUDGMENTS:
        raise RuntimeError(
            f'Collection stopped at {total_saved:,}; expected {TARGET_JUDGMENTS:,}.'
        )

    print('Collection complete ✓')
    print('Total unique judgments:', f'{total_saved:,}')

COLLECTION_MANIFEST_PATH.write_text(
    json.dumps(
        {
            'target_judgments': TARGET_JUDGMENTS,
            'collection_shard_size': COLLECTION_SHARD_SIZE,
            'shuffle_buffer': STREAM_SHUFFLE_BUFFER,
            'random_seed': RANDOM_SEED,
            'dataset': DATASET_NAME,
            'config': DATASET_CONFIG,
            'split': DATASET_SPLIT,
        },
        indent=2
    )
)


Collection progress: 0/100,000


README.md: 0.00B [00:00, ?B/s]


Saved: judgments_000.parquet

Saved: judgments_001.parquet

Saved: judgments_002.parquet

Saved: judgments_003.parquet

Saved: judgments_004.parquet

Saved: judgments_005.parquet

Saved: judgments_006.parquet

Saved: judgments_007.parquet

Saved: judgments_008.parquet

Saved: judgments_009.parquet

Saved: judgments_010.parquet

Saved: judgments_011.parquet

Saved: judgments_012.parquet

Saved: judgments_013.parquet

Saved: judgments_014.parquet

Saved: judgments_015.parquet

Saved: judgments_016.parquet

Saved: judgments_017.parquet

Saved: judgments_018.parquet

Saved: judgments_019.parquet
Collection complete ✓
Total unique judgments: 100,000


210

## 04 Load the collected corpus

In [10]:
collection_files = sorted(BASE_DIR.glob('judgments_*.parquet'))

if not collection_files:
    raise FileNotFoundError(
        'No judgment shards found. Run the dataset collection stage first.'
    )

df = pd.concat(
    [pd.read_parquet(path) for path in collection_files],
    ignore_index=True,
)

print('Rows:', f'{len(df):,}')
print('Unique texts:', f'{df["full_text"].nunique():,}')

if len(df) != TARGET_JUDGMENTS:
    raise ValueError(
        f'Expected {TARGET_JUDGMENTS:,} judgments, found {len(df):,}.'
    )

if df['full_text'].nunique() != TARGET_JUDGMENTS:
    raise ValueError('Collection is not unique by full_text.')

print('Collection validation passed ✓')

Rows: 100,000
Unique texts: 100,000
Collection validation passed ✓


## 05 Data profiling

In [11]:
print('\nCourts:')
print(df['court_code'].value_counts().to_string())

print('\nYears:')
print(
    df['decision_date']
      .astype(str)
      .str[:4]
      .value_counts()
      .sort_index()
      .to_string()
)

print('\nCourt metadata preview:')
print(
    df[['court_code', 'court_name', 'source']]
      .drop_duplicates()
      .sort_values('court_code')
      .to_string(index=False)
)

print('\nLength statistics:')
print(
    df['text_length'].describe().to_string()
)


Courts:
court_code
18_6     21498
19_16    19772
10_8     19679
1_12     10471
24_17     8555
22_18     6961
16_20     3289
14_25     2268
17_21     1738
23_23     1542
27_1      1037
20_7      1014
36_29      663
3_22       385
32_4       343
11_24      287
5_15       162
21_11       85
33_10       80
8_9         66
7_26        63
2_5         33
29_3         7
9_13         2

Years:
decision_date
1996     2583
1998     2993
1999     2646
2000      249
2001      931
2002       71
2008     1109
2009       45
2012      929
2013        5
2015     7452
2016       97
2017       27
2018        7
2019     1288
2020    27933
2021      152
2022      253
2023    13960
2024      945
2025    36319
2026        6

Court metadata preview:
court_code court_name source
      10_8                hc
     11_24                hc
     14_25                hc
     16_20                hc
     17_21                hc
      18_6                hc
     19_16                hc
      1_12                hc
    

## 06 Data quality

In [12]:
def control_count(text):
    return len(
        re.findall(
            r'[\x00-\x08\x0b\x0c\x0e-\x1f]',
            str(text)
        )
    )

df['control_chars'] = df['full_text'].apply(control_count)
bad = df[df['control_chars'] > 0].copy()

print('Documents with control characters:', len(bad))
print(
    bad[['cnr', 'court_code', 'decision_date', 'text_length', 'control_chars']]
      .sort_values('control_chars', ascending=False)
      .head(20)
      .to_string(index=False)
)

print('\n=== OVERALL ===')
print('Total documents:', f'{len(df):,}')
print('Affected:', f'{len(bad):,}')
print('Affected %:', round(len(bad) / len(df) * 100, 2))

Documents with control characters: 762
                          cnr court_code decision_date  text_length  control_chars
PHHC010734172014_1_2016-10-26       3_22    2016-10-26        31038          21061
PHHC010780992012_1_2015-10-31       3_22    2015-10-31        26636          16562
PHHC010087552003_1_2016-12-01       3_22    2016-12-01        20299          12981
MLHC010000122011_1_2012-09-05      17_21    2012-09-05        37440          10474
GAHC040002732002_1_2002-08-27       18_6    2002-08-27        50933           9758
GAHC040004262012_1_2012-08-31       18_6    2012-08-31        45571           8833
GAHC040008452001_1_2002-09-11       18_6    2002-09-11        42048           8825
GAHC040009462014_1_2015-01-09       18_6    2015-01-09        43858           8779
GAHC040011742010_1_2012-04-30       18_6    2012-04-30        43976           8772
GAHC040003842010_1_2012-04-30       18_6    2012-04-30        43976           8772
GAHC040008402012_1_2012-09-07       18_6    2012

### corruption example

In [13]:
if len(bad):
    row = bad.sort_values('control_chars', ascending=False).iloc[0]
    print('CNR:', row['cnr'])
    print('Court:', row['court_code'])
    print('Control chars:', row['control_chars'])
    print('\nFirst 1000 characters:')
    print(str(row['full_text'])[:1000])
else:
    print('No control-character corruption detected.')

CNR: PHHC010734172014_1_2016-10-26
Court: 3_22
Control chars: 21061

First 1000 characters:
                                         !!    !" #$ %      !  &$'#$(#       )))))  " * +          '     ,   &        - .# /0 #.  1    ./ 23$ 4  # #      2#   %+5#  ' +#    &         &            6  7  #  $8    8         $ +    &  '  

## 07 Create the cleaned corpus

In [14]:
# Keep the full 100k collection and clean control characters during normalization.
# We do not silently drop documents from the final corpus.
df_clean = df.copy()

print('Original:', f'{len(df):,}')
print('Retained:', f'{len(df_clean):,}')
print('Removed during this stage: 0')


Original: 100,000
Retained: 100,000
Removed during this stage: 0


In [15]:
def normalize_text(text):
    text = str(text)
    text = re.sub(
        r'[\x00-\x08\x0b\x0c\x0e-\x1f]',
        ' ',
        text
    )
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' *\n *', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

df_clean['clean_text'] = df_clean['full_text'].apply(normalize_text)
df_clean['clean_length'] = df_clean['clean_text'].str.len()

if (df_clean['clean_text'].str.len() < MIN_CHUNK_LENGTH).any():
    print(
        'Documents below minimum chunk length:',
        int((df_clean['clean_text'].str.len() < MIN_CHUNK_LENGTH).sum())
    )

CLEAN_PATH = BASE_DIR / 'legal_judgments_clean.parquet'
df_clean.to_parquet(CLEAN_PATH, index=False)

print('Saved:', CLEAN_PATH)
print('Usable documents:', f'{len(df_clean):,}')


Saved: /kaggle/working/legalrag_100k/legal_judgments_clean.parquet
Usable documents: 100,000


## 08 Create chunks

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNKS_PATH = BASE_DIR / 'legal_chunks.parquet'

if CHUNKS_PATH.exists():
    chunks_df = pd.read_parquet(CHUNKS_PATH)
    print('Loaded existing chunks:', f'{len(chunks_df):,}')
else:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=['\n\n', '\n', '. ', ' ', '']
    )

    chunks = []

    for _, row in tqdm(
        df_clean.iterrows(),
        total=len(df_clean),
        desc='Chunking judgments'
    ):
        split_chunks = splitter.split_text(
            row['clean_text']
        )

        for i, chunk in enumerate(split_chunks):
            if len(chunk) < MIN_CHUNK_LENGTH:
                continue

            chunks.append({
                'chunk_id': f"{row['cnr']}_{i}",
                'cnr': row['cnr'],
                'court_code': row['court_code'],
                'decision_date': row['decision_date'],
                'case_type': row['case_type'],
                'title': row['title'],
                'chunk_index': i,
                'text': chunk,
            })

    chunks_df = pd.DataFrame(chunks)
    chunks_df.reset_index(drop=True, inplace=True)
    chunks_df.to_parquet(CHUNKS_PATH, index=False)

    del chunks
    gc.collect()

    print('Saved chunks:', f'{len(chunks_df):,}')

print('Chunks:', f'{len(chunks_df):,}')
print(chunks_df.head(3)[['chunk_id', 'chunk_index', 'text']].to_string(index=False))
print('\nChunk lengths:')
print(chunks_df['text'].str.len().describe().to_string())


Chunking judgments:   0%|          | 0/100000 [00:00<?, ?it/s]

Saved chunks: 538,079
Chunks: 538,079
                       chunk_id  chunk_index                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

# Evaluation Dataset

In [17]:
EVAL_DOC_PATH = BASE_DIR / 'evaluation_documents.parquet'

if EVAL_DOC_PATH.exists():
    eval_docs = pd.read_parquet(EVAL_DOC_PATH)
    print('Loaded evaluation documents:', len(eval_docs))
else:
    eligible = df_clean[
        (df_clean['clean_length'] >= 500) &
        (df_clean['clean_length'] <= EVAL_DOCUMENT_MAX_CHARS)
    ].copy()

    if len(eligible) < EVAL_DOCUMENTS:
        raise ValueError(
            f'Only {len(eligible):,} documents satisfy the benchmark length bounds; '
            f'{EVAL_DOCUMENTS:,} are required.'
        )

    # Shuffle first, then take documents round-robin across courts.
    # This improves court coverage without changing retrieval itself.
    eligible = eligible.sample(
        frac=1.0,
        random_state=RANDOM_SEED
    ).reset_index(drop=True)
    eligible['_court_round'] = eligible.groupby('court_code').cumcount()

    eval_docs = (
        eligible
        .sort_values(['_court_round', 'court_code'])
        .head(EVAL_DOCUMENTS)
        .drop(columns=['_court_round'])
        .copy()
    )

    # Fixed benchmark composition.
    question_types = []
    for qtype, count in QUESTION_TYPE_TARGETS.items():
        question_types.extend([qtype] * count)

    rng = random.Random(RANDOM_SEED)
    rng.shuffle(question_types)

    eval_docs['target_question_type'] = question_types

    eval_docs.to_parquet(EVAL_DOC_PATH, index=False)
    print('Saved evaluation documents:', len(eval_docs))

if len(eval_docs) != EVAL_DOCUMENTS:
    raise ValueError(
        f'Expected {EVAL_DOCUMENTS} evaluation documents, found {len(eval_docs)}.'
    )

print('\nEvaluation questions by target type:')
print(
    eval_docs['target_question_type']
    .value_counts()
    .sort_index()
    .to_string()
)

print('\nEvaluation documents by court:')
print(
    eval_docs['court_code']
    .value_counts()
    .to_string()
)


Saved evaluation documents: 500

Evaluation questions by target type:
target_question_type
fact               100
legal_provision     80
multi_hop           50
outcome            120
reasoning          150

Evaluation documents by court:
court_code
10_8     23
11_24    23
14_25    23
16_20    23
17_21    23
18_6     23
19_16    23
1_12     22
20_7     22
21_11    22
22_18    22
23_23    22
24_17    22
27_1     22
2_5      22
32_4     22
5_15     22
33_10    22
36_29    22
3_22     22
8_9      22
7_26     22
29_3      7
9_13      2


## 09 OmniRoute setup

All benchmark generation, RAG generation, and RAG evaluation calls use the same OmniRoute Combo **`new`**.


In [1]:
# ============================================================
# OmniRoute — Setup & Connection Test
# ============================================================

from kaggle_secrets import UserSecretsClient
from openai import OpenAI

# ------------------------------------------------------------
# OmniRoute configuration
# ------------------------------------------------------------

OMNIROUTE_BASE_URL = "https://mate-statutory-temp-planners.trycloudflare.com/v1"
EVAL_MODEL = "kaggle"

# ------------------------------------------------------------
# API key from Kaggle Secrets
# ------------------------------------------------------------

secrets = UserSecretsClient()

omniroute_api_key = secrets.get_secret("OMNIROUTE_API_KEY")

if not omniroute_api_key:
    raise RuntimeError(
        "OMNIROUTE_API_KEY is missing from Kaggle Secrets."
    )

# ------------------------------------------------------------
# Create OpenAI-compatible OmniRoute client
# ------------------------------------------------------------

client = OpenAI(
    api_key=omniroute_api_key.strip(),
    base_url=OMNIROUTE_BASE_URL
)

print("OmniRoute client ready ✓")
print("Model requested:", EVAL_MODEL)
print("Base URL:", OMNIROUTE_BASE_URL)

# ------------------------------------------------------------
# Connection test
# ------------------------------------------------------------

response = client.chat.completions.create(
    model=EVAL_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: OmniRoute works"
        }
    ],
    temperature=0
)

print()
print("Connection test response:")
print(response.choices[0].message.content.strip())

print()
print("Server model:", response.model)

OmniRoute client ready ✓
Model requested: kaggle
Base URL: https://mate-statutory-temp-planners.trycloudflare.com/v1

Connection test response:
OmniRoute works

Server model: minimax-m2.5


## 10 10-question pilot

In [42]:
PROMPT_TEMPLATE = '''
You are creating a high-quality evaluation benchmark for a legal RAG system.

Using ONLY the Indian court judgment below, create ONE question that:
1. Requires understanding the judgment.
2. Can be answered from the judgment.
3. Is not answerable merely from the case number, date, judge, or party names.
4. Focuses on legal reasoning, relevant facts, legal provision/application, or final outcome.
5. Does not introduce information not present in the judgment.

Return ONLY valid JSON in exactly this format:
{
  "question": "...",
  "reference_answer": "...",
  "question_type": "fact|reasoning|legal_provision|outcome|multi_hop",
  "supporting_text": "Exact or near-exact passage from the judgment supporting the answer."
}

The supporting_text must be taken from the judgment itself.

JUDGMENT:
'''

PILOT_PATH = BASE_DIR / 'eval_pilot_10.json'
pilot_docs = eval_docs.head(PILOT_DOCUMENTS).copy()

PILOT_TYPES = [
    'fact', 'reasoning', 'legal_provision', 'outcome', 'multi_hop',
    'reasoning', 'outcome', 'fact', 'legal_provision', 'reasoning'
]

def parse_json_response(text):
    text = str(text).strip()

    if text.startswith('```'):
        lines = text.splitlines()
        lines = lines[1:] if lines and lines[0].strip().startswith('```') else lines
        lines = lines[:-1] if lines and lines[-1].strip() == '```' else lines
        text = '\n'.join(lines).strip()

    start = text.find('{')
    end = text.rfind('}')

    if start < 0 or end <= start:
        raise ValueError('No JSON object found in model response.')

    return json.loads(text[start:end + 1])

if PILOT_PATH.exists():
    pilot_results = json.loads(PILOT_PATH.read_text())
    print('Loaded existing pilot:', len(pilot_results))
else:
    pilot_results = []

    for i, (_, row) in enumerate(pilot_docs.iterrows(), start=1):
        target_type = PILOT_TYPES[i - 1]

        print(f'Generating pilot {i}/{PILOT_DOCUMENTS} | {row["cnr"]}')

        prompt = (
            PROMPT_TEMPLATE
            + f'\nTARGET QUESTION TYPE: {target_type}\n\n'
            + str(row['clean_text'])
        )

        response = client.chat.completions.create(
            model=EVAL_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0
        )

        item = parse_json_response(
            response.choices[0].message.content
        )

        required = {
            'question', 'reference_answer',
            'question_type', 'supporting_text'
        }

        if not required.issubset(item):
            raise ValueError(
                f'Pilot item is missing required fields: {required - set(item)}'
            )

        if item['question_type'] != target_type:
            raise ValueError(
                f'Pilot question type mismatch: expected {target_type}, '
                f'got {item["question_type"]}.'
            )

        item['cnr'] = str(row['cnr'])
        pilot_results.append(item)

        PILOT_PATH.write_text(
            json.dumps(pilot_results, indent=2, ensure_ascii=False)
        )

print('Pilot generated:', len(pilot_results))


NameError: name 'eval_docs' is not defined

In [22]:
for i, item in enumerate(pilot_results, start=1):
    print('=' * 90)
    print(f'{i}. {item["question"]}')
    print('TYPE:', item['question_type'])
    print('ANSWER:', item['reference_answer'])
    print('SUPPORT:', item['supporting_text'][:500])


1. What are the specific allegations leveled against the appellant according to the prosecution case?
TYPE: fact
ANSWER: According to the prosecution case, it is alleged that the appellant abused the informant by caste name and assaulted him.
SUPPORT: As per prosecution case, it is alleged that this appellant abused informant by caste name and assaulted him.
2. Why did the High Court of Sikkim direct the appellant to approach the Debt Recovery Tribunal (DRT) at Siliguri instead of deciding the recovery dispute directly, despite the Bank's admission that the property was never mortgaged?
TYPE: reasoning
ANSWER: Although the Bank conceded that the property was not mortgaged by the appellant, the Bank's affidavit disclosed that the property had been attached and subjected to a sale order by the DRT to recover dues from the actual borrower, and these DRT orders were never challenged by the appellant before any competent authority. Consequently, considering the Bank's assertions and the fac

## 11 Generate remaining questions in restartable calls

In [ ]:
BENCHMARK_PROMPT = '''
You are generating a high-quality benchmark for a legal RAG system.

Create exactly ONE evaluation question from the supplied Indian court judgment.

Rules:
- Use ONLY the supplied judgment.
- The question must require understanding, not simple metadata lookup.
- The question should test the requested question type.
- The reference answer must be supported by the judgment.
- supporting_text must be copied or closely extracted from the judgment.
- Do not introduce information that is not present in the judgment.

Return ONLY valid JSON in exactly this format:
{
  "question": "...",
  "reference_answer": "...",
  "question_type": "fact|reasoning|legal_provision|outcome|multi_hop",
  "supporting_text": "..."
}

TARGET QUESTION TYPE:
'''

CANDIDATES_PATH = BASE_DIR / 'eval_candidates.json'

if CANDIDATES_PATH.exists():
    candidates = json.loads(CANDIDATES_PATH.read_text())
    print('Existing candidates:', len(candidates))
else:
    candidates = []

existing_cnrs = {str(x['cnr']) for x in candidates}

remaining_docs = eval_docs[
    ~eval_docs['cnr'].astype(str).isin(existing_cnrs)
].copy()

print('Existing candidates:', len(candidates))
print('Remaining judgments:', len(remaining_docs))

MAX_RETRIES = 3

for i, (_, row) in enumerate(remaining_docs.iterrows(), start=1):

    cnr = str(row['cnr'])
    target_type = str(row['target_question_type'])

    print('\n' + '-' * 70)
    print(f'Generating benchmark {len(candidates) + 1}/{EVAL_DOCUMENTS}')
    print(f'CNR: {cnr}')
    print(f'Target type: {target_type}')

    prompt = (
        BENCHMARK_PROMPT
        + f' {target_type}\n\n'
        + f'CNR: {cnr}\n'
        + 'JUDGMENT:\n'
        + str(row['clean_text'])
    )

    success = False

    for attempt in range(1, MAX_RETRIES + 1):

        print(f'Attempt {attempt}/{MAX_RETRIES}')

        try:
            response = client.chat.completions.create(
                model=EVAL_MODEL,
                messages=[
                    {
                        'role': 'user',
                        'content': prompt
                    }
                ],
                temperature=0
            )

            raw_response = response.choices[0].message.content

            if not raw_response or not raw_response.strip():
                raise ValueError('Empty model response.')

            item = parse_json_response(raw_response)

            required = {
                'question',
                'reference_answer',
                'question_type',
                'supporting_text'
            }

            if not required.issubset(item):
                raise ValueError(
                    f'Missing required fields: '
                    f'{required - set(item)}'
                )

            if item['question_type'] != target_type:
                raise ValueError(
                    f'Question type mismatch for {cnr}: '
                    f'expected {target_type}, '
                    f'got {item["question_type"]}.'
                )

            # Basic content validation
            for field in [
                'question',
                'reference_answer',
                'supporting_text'
            ]:
                if not str(item[field]).strip():
                    raise ValueError(
                        f'Empty required field: {field}'
                    )

            item['cnr'] = cnr
            item['target_question_type'] = target_type

            candidates.append(item)

            CANDIDATES_PATH.write_text(
                json.dumps(
                    candidates,
                    indent=2,
                    ensure_ascii=False
                )
            )

            print(
                f'SUCCESS | Progress: '
                f'{len(candidates)}/{EVAL_DOCUMENTS}'
            )
            print('Checkpoint saved:', CANDIDATES_PATH)

            success = True
            break

        except Exception as e:

            print(
                f'Attempt {attempt} failed: '
                f'{type(e).__name__}: {e}'
            )

            if attempt < MAX_RETRIES:
                print(
                    'Retrying the same judgment...'
                )
            else:
                print()
                print('BENCHMARK GENERATION ERROR')
                print('CNR:', cnr)
                print(
                    'Maximum retries reached.'
                )
                print(
                    'Completed candidates remain saved.'
                )
                raise

    if not success:
        raise RuntimeError(
            f'Failed to generate benchmark for {cnr}.'
        )

print('\nFinal candidates saved:', len(candidates))

if len(candidates) != EVAL_DOCUMENTS:
    raise RuntimeError(
        f'Expected {EVAL_DOCUMENTS} candidates, '
        f'found {len(candidates)}.'
    )

Existing candidates: 181
Existing candidates: 181
Remaining judgments: 319

----------------------------------------------------------------------
Generating benchmark 182/500
CNR: PHHC010839062016_1_2016-01-07
Target type: outcome
Attempt 1/3
SUCCESS | Progress: 182/500
Checkpoint saved: /kaggle/working/legalrag_100k/eval_candidates.json

----------------------------------------------------------------------
Generating benchmark 183/500
CNR: UKHC010009112001_1_2001-07-04
Target type: outcome
Attempt 1/3
SUCCESS | Progress: 183/500
Checkpoint saved: /kaggle/working/legalrag_100k/eval_candidates.json

----------------------------------------------------------------------
Generating benchmark 184/500
CNR: DLHC010593602005_1_2017-08-31
Target type: multi_hop
Attempt 1/3
SUCCESS | Progress: 184/500
Checkpoint saved: /kaggle/working/legalrag_100k/eval_candidates.json

----------------------------------------------------------------------
Generating benchmark 185/500
CNR: RJHC020257182007_1_

## 12 Validate gold evidence

In [ ]:
def normalize_eval_text(text):
    return re.sub(r'\s+', ' ', str(text)).strip().lower()

chunks_by_cnr = {
    str(cnr): group.reset_index(drop=True)
    for cnr, group in chunks_df.groupby(chunks_df['cnr'].astype(str), sort=False)
}

full_text_by_cnr = {
    str(row['cnr']): str(row['clean_text'])
    for _, row in df_clean.iterrows()
}

def find_gold_chunks(cnr, supporting_text):

    cnr = str(cnr)
    support = normalize_eval_text(supporting_text)

    if not support or cnr not in chunks_by_cnr:
        return []

    full_text = normalize_eval_text(
        full_text_by_cnr.get(cnr, '')
    )

    doc_chunks = chunks_by_cnr[cnr]
    start = full_text.find(support)

    if start >= 0:
        end = start + len(support)
        hits = []

        for row in doc_chunks.itertuples(index=False):
            chunk_text = normalize_eval_text(row.text)
            chunk_start = full_text.find(chunk_text)

            if chunk_start < 0:
                continue

            chunk_end = chunk_start + len(chunk_text)

            if chunk_start < end and chunk_end > start:
                hits.append(str(row.chunk_id))

        if hits:
            return list(dict.fromkeys(hits))

    words = support.split()

    for window in (60, 40, 25, 15, 10):
        if len(words) < window:
            continue

        for i in range(len(words) - window + 1):
            phrase = ' '.join(words[i:i + window])

            for row in doc_chunks.itertuples(index=False):
                if phrase in normalize_eval_text(row.text):
                    return [str(row.chunk_id)]

    return []

if not CANDIDATES_PATH.exists():
    raise FileNotFoundError(
        'eval_candidates.json was not created.'
    )

candidates = json.loads(
    CANDIDATES_PATH.read_text()
)

validated = []

for item in tqdm(
    candidates,
    desc='Validating gold evidence'
):
    gold_chunks = find_gold_chunks(
        item['cnr'],
        item['supporting_text']
    )

    item['gold_chunk_ids'] = gold_chunks

    if not gold_chunks:
        print('No gold chunk found:', item['cnr'])

    validated.append(item)

missing_gold = [
    item for item in validated
    if not item.get('gold_chunk_ids')
]

if missing_gold:
    raise ValueError(
        f'{len(missing_gold)} benchmark items have no supporting chunk. '
        'Do not continue until benchmark evidence is valid.'
    )

final_eval = []
seen = set()

for item in validated:
    cnr = str(item['cnr'])
    if cnr not in seen:
        final_eval.append(item)
        seen.add(cnr)

GOLD_EVAL_PATH.write_text(
    json.dumps(final_eval, indent=2, ensure_ascii=False)
)

print('Candidates:', len(candidates))
print('With gold evidence:', len(final_eval))
print('Gold evaluation saved:', GOLD_EVAL_PATH)

if len(final_eval) != EVAL_DOCUMENTS:
    raise ValueError(
        f'Expected {EVAL_DOCUMENTS} grounded evaluation questions, '
        f'found {len(final_eval)}.'
    )


## 13 Load the locked evaluation benchmark

In [ ]:
final_eval = json.loads(
    GOLD_EVAL_PATH.read_text()
)

print('Evaluation questions:', len(final_eval))
print(
    'With gold evidence:',
    sum(bool(x.get('gold_chunk_ids')) for x in final_eval)
)

if len(final_eval) != EVAL_DOCUMENTS:
    raise ValueError(
        f'Expected {EVAL_DOCUMENTS} questions, found {len(final_eval)}.'
    )


## Restart 

In [9]:
# ============================================================
# Restore LegalRAG 100k experiment from Kaggle Dataset
# Continue from BM25 stage
# ============================================================

from pathlib import Path
import shutil
import json

SOURCE_DIR = Path(
    "/kaggle/input/datasets/"
    "siddharthdongardive/"
    "legalrag-dense-checkpoint/"
    "legalrag_100k_backup/"
    "legalrag_100k"
)

BASE_DIR = Path("/kaggle/working/legalrag_100k")

print("=" * 80)
print("RESTORING LEGALRAG 100K EXPERIMENT")
print("=" * 80)

if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"Backup directory not found:\n{SOURCE_DIR}"
    )

print("Backup source:")
print(SOURCE_DIR)

# Remove any incomplete working copy
if BASE_DIR.exists():
    print("\nRemoving existing working directory...")
    shutil.rmtree(BASE_DIR)

# Copy the complete experiment state
shutil.copytree(
    SOURCE_DIR,
    BASE_DIR
)

print("\nRestored to:")
print(BASE_DIR)

# ------------------------------------------------------------
# Verify important artifacts
# ------------------------------------------------------------

expected_files = [
    "eval_candidates.json",
    "gold_eval.json",
    "experiment_state.json",
    "MANIFEST.json",
]

print("\nChecking important files:")

for filename in expected_files:

    path = BASE_DIR / filename

    if path.exists():
        print(f"✓ {filename}")
    else:
        print(f"⚠ Missing: {filename}")

# ------------------------------------------------------------
# Load experiment state
# ------------------------------------------------------------

state_path = BASE_DIR / "experiment_state.json"

if state_path.exists():

    experiment_state = json.loads(
        state_path.read_text()
    )

    print("\nExperiment state:")
    print(
        json.dumps(
            experiment_state,
            indent=2
        )
    )

# ------------------------------------------------------------
# Verify benchmark
# ------------------------------------------------------------

gold_path = BASE_DIR / "gold_eval.json"

if gold_path.exists():

    final_eval = json.loads(
        gold_path.read_text()
    )

    print("\nFinal gold benchmark:")
    print(f"Questions: {len(final_eval)}")

    if len(final_eval) != 497:
        raise ValueError(
            f"Expected 497 validated questions, "
            f"found {len(final_eval)}"
        )

    print("✓ 497 validated gold-supported questions")

print("\n" + "=" * 80)
print("LEGALRAG 100K RESTORED ✓")
print("NEXT STAGE: BM25")
print("=" * 80)

RESTORING LEGALRAG 100K EXPERIMENT
Backup source:
/kaggle/input/datasets/siddharthdongardive/legalrag-dense-checkpoint/legalrag_100k_backup/legalrag_100k

Removing existing working directory...

Restored to:
/kaggle/working/legalrag_100k

Checking important files:
✓ eval_candidates.json
✓ gold_eval.json
✓ experiment_state.json
✓ MANIFEST.json

Experiment state:
{
  "project": "LegalRAG",
  "experiment": "100k final experiment",
  "workspace": "/kaggle/working/legalrag_100k",
  "dataset_size": 100000,
  "benchmark_candidates": 500,
  "final_gold_questions": 497,
  "next_stage": "BM25",
  "retrieval_plan": {
    "bm25": true,
    "dense": true,
    "hybrid_rrf": true,
    "reranker": true,
    "candidate_pool": 50
  },
  "chunking": {
    "chunk_size": 1200,
    "chunk_overlap": 200,
    "min_chunk_length": 100
  },
  "generation": {
    "provider": "OmniRoute",
    "model": "kaggle"
  }
}

Final gold benchmark:
Questions: 497
✓ 497 validated gold-supported questions

LEGALRAG 100K RESTO

In [13]:
# ============================================================
# BM25 Retrieval — 100k Final Experiment
# ============================================================

import json
import pickle
import re
import time
from pathlib import Path

import pandas as pd
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE_DIR = Path("/kaggle/working/legalrag_100k")

GOLD_EVAL_PATH = BASE_DIR / "gold_eval.json"
CHUNKS_PATH = BASE_DIR / "legal_chunks.parquet"

BM25_INDEX_PATH = BASE_DIR / "bm25.pkl"
BM25_RESULTS_PATH = BASE_DIR / "bm25_results.json"


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

BM25_TOP_K = 50

print("=" * 80)
print("LEGALRAG — BM25 RETRIEVAL")
print("=" * 80)

print("Workspace:", BASE_DIR)
print("Chunk file:", CHUNKS_PATH)
print("BM25 top-K:", BM25_TOP_K)


# ------------------------------------------------------------
# Validate paths
# ------------------------------------------------------------

if not BASE_DIR.exists():
    raise FileNotFoundError(
        f"Workspace not found:\n{BASE_DIR}"
    )

if not GOLD_EVAL_PATH.exists():
    raise FileNotFoundError(
        f"Gold evaluation file not found:\n{GOLD_EVAL_PATH}"
    )

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(
        f"Chunk file not found:\n{CHUNKS_PATH}"
    )


# ------------------------------------------------------------
# Load final gold benchmark
# ------------------------------------------------------------

final_eval = json.loads(
    GOLD_EVAL_PATH.read_text()
)

if len(final_eval) != 497:
    raise ValueError(
        f"Expected 497 final evaluation records, "
        f"found {len(final_eval)}."
    )

print("Evaluation questions:", len(final_eval))


# ------------------------------------------------------------
# Load chunk corpus
# ------------------------------------------------------------

print("\nLoading legal chunks...")

chunks_df = pd.read_parquet(
    CHUNKS_PATH
).reset_index(drop=True)

required_columns = {
    "chunk_id",
    "cnr",
    "text"
}

missing_columns = (
    required_columns -
    set(chunks_df.columns)
)

if missing_columns:
    raise ValueError(
        f"legal_chunks.parquet is missing required columns: "
        f"{missing_columns}"
    )

print("Chunks:", len(chunks_df))
print("Columns:", list(chunks_df.columns))


# ------------------------------------------------------------
# Legal-aware BM25 tokenizer
# ------------------------------------------------------------

def bm25_tokenize(text):
    """
    Lightweight legal-aware tokenizer.

    Preserves useful legal forms such as:
        156(3)
        Section
        IPC
        CrPC
        Article-21
        WP(C)
    """

    text = str(text).lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    tokens = re.findall(
        r"[a-z0-9]+(?:[-_/().][a-z0-9]+)*",
        text
    )

    return tokens


# ------------------------------------------------------------
# Build or load BM25 index
# ------------------------------------------------------------

if BM25_INDEX_PATH.exists():

    print("\nExisting BM25 index found.")
    print("Loading:", BM25_INDEX_PATH)

    with BM25_INDEX_PATH.open("rb") as f:
        bm25_data = pickle.load(f)

    bm25 = bm25_data["bm25"]
    indexed_chunk_ids = bm25_data["chunk_ids"]

    if len(indexed_chunk_ids) != len(chunks_df):
        raise ValueError(
            "Saved BM25 index does not match "
            "the current legal_chunks.parquet."
        )

    current_chunk_ids = (
        chunks_df["chunk_id"]
        .astype(str)
        .tolist()
    )

    if indexed_chunk_ids != current_chunk_ids:
        raise ValueError(
            "Saved BM25 index chunk order does not match "
            "legal_chunks.parquet."
        )

    print("BM25 index loaded ✓")

else:

    print("\nNo BM25 index found.")
    print("Building BM25 index...")

    build_start = time.time()

    tokenized_corpus = [
        bm25_tokenize(text)
        for text in tqdm(
            chunks_df["text"],
            desc="Tokenizing chunks"
        )
    ]

    bm25 = BM25Okapi(
        tokenized_corpus
    )

    bm25_data = {
        "bm25": bm25,
        "chunk_ids": (
            chunks_df["chunk_id"]
            .astype(str)
            .tolist()
        )
    }

    with BM25_INDEX_PATH.open(
        "wb"
    ) as f:
        pickle.dump(
            bm25_data,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    build_time = time.time() - build_start

    print(
        f"BM25 index built and saved ✓ "
        f"({build_time / 60:.2f} min)"
    )

    del tokenized_corpus


# ------------------------------------------------------------
# Run BM25 retrieval
# ------------------------------------------------------------

print("\nRunning BM25 retrieval...")

retrieval_start = time.time()

bm25_results = {}

for item in tqdm(
    final_eval,
    desc="BM25 retrieval"
):

    cnr = str(item["cnr"])
    question = str(item["question"])

    query_tokens = bm25_tokenize(
        question
    )

    scores = bm25.get_scores(
        query_tokens
    )

    # Top 50 candidates
    top_indices = (
        scores.argsort()[-BM25_TOP_K:][::-1]
    )

    results = []

    for rank, chunk_index in enumerate(
        top_indices,
        start=1
    ):

        chunk_index = int(chunk_index)

        row = chunks_df.iloc[
            chunk_index
        ]

        results.append({
            "rank": rank,
            "chunk_index": chunk_index,
            "chunk_id": str(row["chunk_id"]),
            "cnr": str(row["cnr"]),
            "bm25_score": float(
                scores[chunk_index]
            )
        })

    bm25_results[cnr] = results


# ------------------------------------------------------------
# Save BM25 retrieval results
# ------------------------------------------------------------

BM25_RESULTS_PATH.write_text(
    json.dumps(
        bm25_results,
        indent=2,
        ensure_ascii=False
    )
)

retrieval_time = (
    time.time() -
    retrieval_start
)

print()
print("=" * 80)
print("BM25 RETRIEVAL COMPLETE ✓")
print("=" * 80)

print(
    "Evaluation questions:",
    len(bm25_results)
)

print(
    "Candidates per question:",
    BM25_TOP_K
)

print(
    f"Retrieval time: "
    f"{retrieval_time / 60:.2f} min"
)

print()
print(
    "BM25 index:",
    BM25_INDEX_PATH
)

print(
    "BM25 results:",
    BM25_RESULTS_PATH
)

print("=" * 80)

LEGALRAG — BM25 RETRIEVAL
Workspace: /kaggle/working/legalrag_100k
Chunk file: /kaggle/working/legalrag_100k/legal_chunks.parquet
BM25 top-K: 50
Evaluation questions: 497

Loading legal chunks...
Chunks: 538079
Columns: ['chunk_id', 'cnr', 'court_code', 'decision_date', 'case_type', 'title', 'chunk_index', 'text']

No BM25 index found.
Building BM25 index...


Tokenizing chunks:   0%|          | 0/538079 [00:00<?, ?it/s]

BM25 index built and saved ✓ (1.80 min)

Running BM25 retrieval...


BM25 retrieval:   0%|          | 0/497 [00:00<?, ?it/s]


BM25 RETRIEVAL COMPLETE ✓
Evaluation questions: 497
Candidates per question: 50
Retrieval time: 46.71 min

BM25 index: /kaggle/working/legalrag_100k/bm25.pkl
BM25 results: /kaggle/working/legalrag_100k/bm25_results.json


# Experiment 1 — BM25

In [11]:
from rank_bm25 import BM25Okapi

corpus = (
    chunks_df['text']
    .fillna('')
    .astype(str)
    .tolist()
)

LEGAL_TOKEN_PATTERN = re.compile(
    r'[A-Za-z]+\([A-Za-z0-9]+\)|'
    r'\d+\([A-Za-z0-9]+\)|'
    r'[A-Za-z0-9]+(?:[-/][A-Za-z0-9]+)*'
)

def bm25_tokenize(text):
    return LEGAL_TOKEN_PATTERN.findall(
        str(text).lower()
    )

tokenized_corpus = [
    bm25_tokenize(text)
    for text in corpus
]

bm25 = BM25Okapi(tokenized_corpus)
print('Indexed chunks:', f'{len(corpus):,}')


NameError: name 'chunks_df' is not defined

In [ ]:
def bm25_search(query, k=RETRIEVAL_TOP_K):

    query_tokens = bm25_tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[-k:][::-1]

    results = chunks_df.iloc[top_indices].copy()
    results['chunk_index'] = [int(x) for x in top_indices]
    results['score'] = [float(scores[x]) for x in top_indices]
    results.reset_index(drop=True, inplace=True)

    return results

test_results = bm25_search(
    final_eval[0]['question'],
    k=10
)

print(
    test_results[
        ['chunk_id', 'chunk_index', 'cnr', 'score', 'text']
    ].to_string(index=False)
)


In [14]:
all_top50 = {}

for i, item in enumerate(final_eval, start=1):

    all_top50[str(item['cnr'])] = bm25_search(
        item['question'],
        k=RETRIEVAL_TOP_K
    )

    if i % 25 == 0 or i == len(final_eval):
        print(
            f'Processed {i}/{len(final_eval)} | BM25 top-{RETRIEVAL_TOP_K}'
        )

BM25_RESULTS_PATH = BASE_DIR / 'bm25_top50.json'
BM25_RESULTS_PATH.write_text(
    json.dumps(
        {cnr: frame.to_dict('records') for cnr, frame in all_top50.items()},
        indent=2
    )
)

print('BM25 retrieval complete ✓')
print('Saved:', BM25_RESULTS_PATH)


NameError: name 'bm25_search' is not defined

In [15]:
import numpy as np

def retrieval_metrics(results_dict, label):
    metrics = {
        'retriever': label,
        'corpus_judgments': TARGET_JUDGMENTS,
        'corpus_chunks': len(chunks_df),
    }

    for k in [1, 3, 5, 10, 25, 50]:
        chunk_hits = []
        judgment_hits = []

        for item in final_eval:
            retrieved = results_dict[str(item['cnr'])][:k]

            retrieved_ids = {
                str(x['chunk_id'])
                for x in (
                    retrieved.to_dict('records')
                    if hasattr(retrieved, 'to_dict')
                    else retrieved
                )
            }

            retrieved_cnrs = {
                str(x['cnr'])
                for x in (
                    retrieved.to_dict('records')
                    if hasattr(retrieved, 'to_dict')
                    else retrieved
                )
            }

            gold_ids = {
                str(x)
                for x in item['gold_chunk_ids']
            }

            chunk_hits.append(
                int(bool(retrieved_ids & gold_ids))
            )
            judgment_hits.append(
                int(str(item['cnr']) in retrieved_cnrs)
            )

        metrics[f'chunk_recall@{k}'] = float(np.mean(chunk_hits))
        metrics[f'judgment_recall@{k}'] = float(np.mean(judgment_hits))

        print(
            f'{label} @ {k}: '
            f'chunk={metrics[f"chunk_recall@{k}"]:.4f} | '
            f'judgment={metrics[f"judgment_recall@{k}"]:.4f}'
        )

    return metrics

bm25_metrics = retrieval_metrics(
    all_top50,
    'BM25'
)

BM25_METRICS_PATH = BASE_DIR / 'bm25_metrics.json'
BM25_METRICS_PATH.write_text(
    json.dumps(bm25_metrics, indent=2)
)

print('Saved:', BM25_METRICS_PATH)


# The retrieval results are checkpointed above, so the large BM25 index
# can be released before loading the dense model.
del bm25
del tokenized_corpus
del corpus
gc.collect()
print('BM25 index released from memory ✓')


KeyError: 'BRHC011179452023_1_2025-11-11'

In [16]:
# ============================================================
# Check BM25 retrieval-results alignment
# ============================================================

print("=" * 80)
print("BM25 RESULTS ALIGNMENT CHECK")
print("=" * 80)

print("final_eval:", len(final_eval))
print("bm25_results:", len(bm25_results))
print("all_top50:", len(all_top50))

expected_cnrs = {
    str(item["cnr"])
    for item in final_eval
}

bm25_cnrs = set(
    str(cnr)
    for cnr in bm25_results.keys()
)

all_top50_cnrs = set(
    str(cnr)
    for cnr in all_top50.keys()
)

missing_from_bm25 = expected_cnrs - bm25_cnrs
missing_from_all_top50 = expected_cnrs - all_top50_cnrs

print()
print(
    "Expected CNRs missing from bm25_results:",
    len(missing_from_bm25)
)

if missing_from_bm25:
    print("Examples:")
    for cnr in list(sorted(missing_from_bm25))[:10]:
        print(" ", cnr)

print()
print(
    "Expected CNRs missing from all_top50:",
    len(missing_from_all_top50)
)

if missing_from_all_top50:
    print("Examples:")
    for cnr in list(sorted(missing_from_all_top50))[:10]:
        print(" ", cnr)

print("=" * 80)

BM25 RESULTS ALIGNMENT CHECK
final_eval: 497
bm25_results: 497
all_top50: 0

Expected CNRs missing from bm25_results: 0

Expected CNRs missing from all_top50: 497
Examples:
  BRHC010049222024_1_2025-11-24
  BRHC010193312025_1_2025-10-10
  BRHC010229152025_1_2025-11-03
  BRHC010398862025_1_2025-12-05
  BRHC010647642024_1_2025-04-04
  BRHC010863252025_1_2025-09-12
  BRHC010875962025_1_2025-09-24
  BRHC010894452025_1_2025-08-30
  BRHC010916272025_1_2025-09-17
  BRHC010958212025_1_2025-11-01


In [17]:
# Use the completed BM25 results for the notebook's
# existing retrieval_metrics() function.

all_top50 = bm25_results

print("=" * 70)
print("BM25 RESULTS CONNECTED TO METRICS")
print("=" * 70)

print("all_top50:", len(all_top50))
print("Expected:", len(final_eval))

if len(all_top50) != len(final_eval):
    raise ValueError(
        f"BM25 result count mismatch: "
        f"{len(all_top50)} results for "
        f"{len(final_eval)} evaluation questions."
    )

print("Alignment check ✓")
print("=" * 70)

BM25 RESULTS CONNECTED TO METRICS
all_top50: 497
Expected: 497
Alignment check ✓


In [18]:
bm25_metrics = retrieval_metrics(
    all_top50,
    'BM25'
)

BM25_METRICS_PATH = BASE_DIR / 'bm25_metrics.json'

BM25_METRICS_PATH.write_text(
    json.dumps(
        bm25_metrics,
        indent=2
    )
)

print('Saved:', BM25_METRICS_PATH)

BM25 @ 1: chunk=0.2133 | judgment=0.4266
BM25 @ 3: chunk=0.3018 | judgment=0.5030
BM25 @ 5: chunk=0.3380 | judgment=0.5332
BM25 @ 10: chunk=0.3903 | judgment=0.5775
BM25 @ 25: chunk=0.4185 | judgment=0.6076
BM25 @ 50: chunk=0.4608 | judgment=0.6398
Saved: /kaggle/working/legalrag_100k/bm25_metrics.json


# Experiment 2 — Dense Retrieval

In [19]:
texts = (
    chunks_df['text']
    .fillna('')
    .astype(str)
    .tolist()
)

num_chunks = len(texts)
print('Chunks:', f'{num_chunks:,}')


Chunks: 538,079


## 14 Load BGE model for two GPUs

In [20]:
from transformers import AutoTokenizer, AutoModel

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError(
        'This notebook expects two CUDA GPUs for dense embeddings.'
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

base_model = AutoModel.from_pretrained(
    MODEL_NAME
).to('cuda:0')

base_model.eval()

model = torch.nn.DataParallel(
    base_model,
    device_ids=[0, 1],
    output_device=0
)

model.eval()

print('Model:', MODEL_NAME)
print('GPUs: 0 + 1')
print('Embedding dimension:', EMBED_DIM)
print('Query instruction enabled:', True)


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: BAAI/bge-base-en-v1.5
GPUs: 0 + 1
Embedding dimension: 768
Query instruction enabled: True


In [21]:
def encode_batch(batch_texts, is_query=False):

    if is_query:
        batch_texts = [
            QUERY_INSTRUCTION + str(text)
            for text in batch_texts
        ]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

    inputs = {
        key: value.to('cuda:0')
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0]

        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

    return embeddings.cpu().numpy().astype(np.float32)

test_embeddings = encode_batch(texts[:8], is_query=False)

print('Test shape:', test_embeddings.shape)
print(
    'First vector norm:',
    np.linalg.norm(test_embeddings[0])
)

if test_embeddings.shape != (8, EMBED_DIM):
    raise RuntimeError(
        f'Unexpected embedding shape: {test_embeddings.shape}'
    )

if not np.isclose(
    np.linalg.norm(test_embeddings[0]),
    1.0,
    atol=1e-3
):
    raise RuntimeError(
        'Embedding normalization test failed.'
    )

query_test = encode_batch(
    [final_eval[0]['question']],
    is_query=True
)

if query_test.shape != (1, EMBED_DIM):
    raise RuntimeError(
        f'Unexpected query embedding shape: {query_test.shape}'
    )

print('Dense model test passed ✓')


Test shape: (8, 768)
First vector norm: 1.0
Dense model test passed ✓


## 15 Checkpointed dense embeddings

In [22]:
EMBED_DIR = BASE_DIR / 'embeddings'
EMBED_DIR.mkdir(parents=True, exist_ok=True)

num_shards = (
    num_chunks + EMBED_SHARD_SIZE - 1
) // EMBED_SHARD_SIZE

print('Embedding shards:', num_shards)
print('Shard size:', EMBED_SHARD_SIZE)


Embedding shards: 108
Shard size: 5000


In [23]:
total_start = time.time()

for shard_id in range(num_shards):

    start = shard_id * EMBED_SHARD_SIZE
    end = min(
        start + EMBED_SHARD_SIZE,
        num_chunks
    )

    shard_path = EMBED_DIR / f'emb_{shard_id:04d}.npy'
    expected_shape = (end - start, EMBED_DIM)

    if shard_path.exists():
        try:
            existing = np.load(
                shard_path,
                mmap_mode='r'
            )

            if existing.shape == expected_shape:
                print(
                    f'[{shard_id + 1}/{num_shards}] '
                    f'SKIP ✓ {start:,}:{end:,}'
                )
                del existing
                continue

            del existing
            shard_path.unlink()

        except Exception:
            if shard_path.exists():
                shard_path.unlink()

    shard_start = time.time()

    print('\n' + '=' * 70)
    print(f'SHARD {shard_id + 1}/{num_shards}')
    print(f'Chunks: {start:,} → {end:,}')
    print('=' * 70)

    shard_texts = texts[start:end]
    outputs = []

    for batch_start in tqdm(
        range(0, len(shard_texts), EMBED_BATCH_SIZE),
        desc=f'Shard {shard_id + 1}',
        unit='batch'
    ):
        batch = shard_texts[
            batch_start:batch_start + EMBED_BATCH_SIZE
        ]

        outputs.append(
            encode_batch(batch, is_query=False)
        )

    embeddings = np.concatenate(
        outputs,
        axis=0
    ).astype(np.float32)

    if embeddings.shape != expected_shape:
        raise ValueError(
            f'Unexpected embedding shape {embeddings.shape}; '
            f'expected {expected_shape}'
        )

    tmp_path = EMBED_DIR / f'emb_{shard_id:04d}.tmp.npy'

    np.save(
        tmp_path,
        embeddings
    )

    os.replace(
        tmp_path,
        shard_path
    )

    elapsed = time.time() - shard_start
    progress = (shard_id + 1) / num_shards * 100

    print(
        f'✓ SAVED {shard_path.name} | shape={embeddings.shape}'
    )
    print(f'Shard time: {elapsed / 60:.2f} min')
    print(
        f'Progress: {shard_id + 1}/{num_shards} '
        f'({progress:.1f}%)'
    )

    del embeddings
    del outputs
    del shard_texts
    gc.collect()

    for gpu_id in range(torch.cuda.device_count()):
        torch.cuda.set_device(gpu_id)
        torch.cuda.empty_cache()

print(
    f'\nAll embedding shards completed. '
    f'Total time: {(time.time() - total_start) / 3600:.2f} hours'
)



SHARD 1/108
Chunks: 0 → 5,000


Shard 1:   0%|          | 0/157 [00:00<?, ?batch/s]

✓ SAVED emb_0000.npy | shape=(5000, 768)
Shard time: 1.17 min
Progress: 1/108 (0.9%)

SHARD 2/108
Chunks: 5,000 → 10,000


Shard 2:   0%|          | 0/157 [00:00<?, ?batch/s]

✓ SAVED emb_0001.npy | shape=(5000, 768)
Shard time: 1.32 min
Progress: 2/108 (1.9%)

SHARD 3/108
Chunks: 10,000 → 15,000


...

✓ SAVED emb_0105.npy | shape=(5000, 768)
Shard time: 1.27 min
Progress: 106/108 (98.1%)

SHARD 107/108
Chunks: 530,000 → 535,000


Shard 107:   0%|          | 0/157 [00:00<?, ?batch/s]

✓ SAVED emb_0106.npy | shape=(5000, 768)
Shard time: 1.28 min
Progress: 107/108 (99.1%)

SHARD 108/108
Chunks: 535,000 → 538,079


Shard 108:   0%|          | 0/97 [00:00<?, ?batch/s]

✓ SAVED emb_0107.npy | shape=(3079, 768)
Shard time: 0.80 min
Progress: 108/108 (100.0%)

All embedding shards completed. Total time: 2.43 hours


## 16 Validate embedding shards

In [24]:
embedding_shards = sorted(
    EMBED_DIR.glob('emb_*.npy')
)

if len(embedding_shards) != num_shards:
    raise ValueError(
        f'Expected {num_shards} embedding shards, found {len(embedding_shards)}'
    )

for shard_id in range(num_shards):

    start = shard_id * EMBED_SHARD_SIZE
    end = min(
        start + EMBED_SHARD_SIZE,
        num_chunks
    )

    shard_path = EMBED_DIR / f'emb_{shard_id:04d}.npy'

    arr = np.load(
        shard_path,
        mmap_mode='r'
    )

    expected_shape = (
        end - start,
        EMBED_DIM
    )

    if arr.shape != expected_shape:
        raise ValueError(
            f'{shard_path.name}: {arr.shape} != {expected_shape}'
        )

    print(
        f'Shard {shard_id + 1}/{num_shards}: {arr.shape} ✓'
    )
    del arr

print('\nAll embedding shards validated successfully ✓')


Shard 1/108: (5000, 768) ✓
Shard 2/108: (5000, 768) ✓
Shard 3/108: (5000, 768) ✓
Shard 4/108: (5000, 768) ✓
Shard 5/108: (5000, 768) ✓
Shard 6/108: (5000, 768) ✓
Shard 7/108: (5000, 768) ✓
Shard 8/108: (5000, 768) ✓
Shard 9/108: (5000, 768) ✓
Shard 10/108: (5000, 768) ✓
Shard 11/108: (5000, 768) ✓
Shard 12/108: (5000, 768) ✓
Shard 13/108: (5000, 768) ✓
Shard 14/108: (5000, 768) ✓
Shard 15/108: (5000, 768) ✓
Shard 16/108: (5000, 768) ✓
Shard 17/108: (5000, 768) ✓
Shard 18/108: (5000, 768) ✓
Shard 19/108: (5000, 768) ✓
Shard 20/108: (5000, 768) ✓
Shard 21/108: (5000, 768) ✓
Shard 22/108: (5000, 768) ✓
Shard 23/108: (5000, 768) ✓
Shard 24/108: (5000, 768) ✓
Shard 25/108: (5000, 768) ✓
Shard 26/108: (5000, 768) ✓
Shard 27/108: (5000, 768) ✓
Shard 28/108: (5000, 768) ✓
Shard 29/108: (5000, 768) ✓
Shard 30/108: (5000, 768) ✓
Shard 31/108: (5000, 768) ✓
Shard 32/108: (5000, 768) ✓
Shard 33/108: (5000, 768) ✓
Shard 34/108: (5000, 768) ✓
Shard 35/108: (5000, 768) ✓
Shard 36/108: (5000, 768) ✓
S

## 17 Build persistent FAISS index

In [25]:
import faiss

FAISS_PATH = BASE_DIR / 'dense.index'
FAISS_META_PATH = BASE_DIR / 'dense_index_metadata.json'

if FAISS_PATH.exists() and FAISS_META_PATH.exists():

    index = faiss.read_index(str(FAISS_PATH))
    metadata = json.loads(FAISS_META_PATH.read_text())

    checks = {
        'num_chunks': index.ntotal == num_chunks,
        'dimension': index.d == EMBED_DIM,
        'model': metadata.get('model') == MODEL_NAME,
        'chunk_count_metadata': metadata.get('num_chunks') == num_chunks,
    }

    if not all(checks.values()):
        raise ValueError(
            f'Existing FAISS index does not match current corpus/model: {checks}'
        )

    print('Existing FAISS index validated and reused ✓')

else:

    index = faiss.IndexFlatIP(EMBED_DIM)

    for shard_id in range(num_shards):

        shard_path = EMBED_DIR / f'emb_{shard_id:04d}.npy'
        embeddings = np.load(shard_path).astype(np.float32)

        index.add(embeddings)

        print(
            f'Added {shard_id + 1}/{num_shards} | vectors={index.ntotal:,}'
        )

        del embeddings
        gc.collect()

    if index.ntotal != num_chunks:
        raise ValueError(
            f'FAISS contains {index.ntotal:,} vectors, expected {num_chunks:,}'
        )

    faiss.write_index(
        index,
        str(FAISS_PATH)
    )

    FAISS_META_PATH.write_text(
        json.dumps(
            {
                'model': MODEL_NAME,
                'embedding_dimension': EMBED_DIM,
                'num_chunks': num_chunks,
                'shard_size': EMBED_SHARD_SIZE,
                'metric': 'inner_product',
                'normalized_embeddings': True,
                'query_instruction': QUERY_INSTRUCTION,
                'corpus_judgments': TARGET_JUDGMENTS,
            },
            indent=2
        )
    )

    print('FAISS index saved ✓')

print('FAISS vectors:', f'{index.ntotal:,}')
print('FAISS dimension:', index.d)


Added 1/108 | vectors=5,000
Added 2/108 | vectors=10,000
Added 3/108 | vectors=15,000
Added 4/108 | vectors=20,000
Added 5/108 | vectors=25,000
Added 6/108 | vectors=30,000
Added 7/108 | vectors=35,000
Added 8/108 | vectors=40,000
Added 9/108 | vectors=45,000
Added 10/108 | vectors=50,000
Added 11/108 | vectors=55,000
Added 12/108 | vectors=60,000
Added 13/108 | vectors=65,000
Added 14/108 | vectors=70,000
Added 15/108 | vectors=75,000
Added 16/108 | vectors=80,000
Added 17/108 | vectors=85,000
Added 18/108 | vectors=90,000
Added 19/108 | vectors=95,000
Added 20/108 | vectors=100,000
Added 21/108 | vectors=105,000
Added 22/108 | vectors=110,000
Added 23/108 | vectors=115,000
Added 24/108 | vectors=120,000
Added 25/108 | vectors=125,000
Added 26/108 | vectors=130,000
Added 27/108 | vectors=135,000
Added 28/108 | vectors=140,000
Added 29/108 | vectors=145,000
Added 30/108 | vectors=150,000
Added 31/108 | vectors=155,000
Added 32/108 | vectors=160,000
Added 33/108 | vectors=165,000
Added 

## 18 Dense retrieval

In [26]:
final_eval = json.loads(
    GOLD_EVAL_PATH.read_text()
)

dense_top50 = {}

for i, item in enumerate(final_eval, start=1):

    query_embedding = encode_batch(
        [str(item['question'])],
        is_query=True
    )

    scores, indices = index.search(
        query_embedding,
        RETRIEVAL_TOP_K
    )

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):
        if idx < 0:
            continue

        results.append({
            'rank': rank,
            'chunk_index': int(idx),
            'chunk_id': str(chunks_df.iloc[idx]['chunk_id']),
            'cnr': str(chunks_df.iloc[idx]['cnr']),
            'score': float(score),
        })

    dense_top50[str(item['cnr'])] = results

    if i % 25 == 0 or i == len(final_eval):
        print(
            f'Processed {i}/{len(final_eval)} | Dense top-{RETRIEVAL_TOP_K}'
        )

DENSE_RESULTS_PATH = BASE_DIR / 'dense_top50.json'
DENSE_RESULTS_PATH.write_text(
    json.dumps(dense_top50, indent=2)
)

print('Dense retrieval complete ✓')
print('Saved:', DENSE_RESULTS_PATH)


Processed 25/497 | Dense top-50
Processed 50/497 | Dense top-50
Processed 75/497 | Dense top-50
Processed 100/497 | Dense top-50
Processed 125/497 | Dense top-50
Processed 150/497 | Dense top-50
Processed 175/497 | Dense top-50
Processed 200/497 | Dense top-50
Processed 225/497 | Dense top-50
Processed 250/497 | Dense top-50
Processed 275/497 | Dense top-50
Processed 300/497 | Dense top-50
Processed 325/497 | Dense top-50
Processed 350/497 | Dense top-50
Processed 375/497 | Dense top-50
Processed 400/497 | Dense top-50
Processed 425/497 | Dense top-50
Processed 450/497 | Dense top-50
Processed 475/497 | Dense top-50
Processed 497/497 | Dense top-50
Dense retrieval complete ✓
Saved: /kaggle/working/legalrag_100k/dense_top50.json


## 19 Dense Recall@K

In [27]:
dense_metrics = retrieval_metrics(
    dense_top50,
    'Dense'
)
dense_metrics['model'] = MODEL_NAME
dense_metrics['embedding_dimension'] = EMBED_DIM

DENSE_METRICS_PATH = BASE_DIR / 'dense_metrics.json'
DENSE_METRICS_PATH.write_text(
    json.dumps(dense_metrics, indent=2)
)

print('Saved:', DENSE_METRICS_PATH)


Dense @ 1: chunk=0.0885 | judgment=0.1972
Dense @ 3: chunk=0.1368 | judgment=0.2535
Dense @ 5: chunk=0.1529 | judgment=0.2736
Dense @ 10: chunk=0.1911 | judgment=0.3340
Dense @ 25: chunk=0.2616 | judgment=0.4125
Dense @ 50: chunk=0.2938 | judgment=0.4467
Saved: /kaggle/working/legalrag_100k/dense_metrics.json


# BM25 vs Dense

In [29]:
bm25_metrics = json.loads(
    (BASE_DIR / 'bm25_metrics.json').read_text()
)
dense_metrics = json.loads(
    (BASE_DIR / 'dense_metrics.json').read_text()
)

comparison = pd.DataFrame({
    'Retriever': ['BM25', 'Dense'],
    'Judgment Recall@1': [
        bm25_metrics['judgment_recall@1'],
        dense_metrics['judgment_recall@1']
    ],
    'Judgment Recall@5': [
        bm25_metrics['judgment_recall@5'],
        dense_metrics['judgment_recall@5']
    ],
    'Judgment Recall@10': [
        bm25_metrics['judgment_recall@10'],
        dense_metrics['judgment_recall@10']
    ],
    'Judgment Recall@50': [
        bm25_metrics['judgment_recall@50'],
        dense_metrics['judgment_recall@50']
    ],
})

print(comparison.to_string(index=False))

RETRIEVAL_COMPARISON_PATH = BASE_DIR / 'bm25_vs_dense.json'
RETRIEVAL_COMPARISON_PATH.write_text(
    comparison.to_json(
        orient='records',
        indent=2
    )
)

print('Saved:', RETRIEVAL_COMPARISON_PATH)


Retriever  Judgment Recall@1  Judgment Recall@5  Judgment Recall@10  Judgment Recall@50
     BM25           0.426559           0.533199            0.577465            0.639839
    Dense           0.197183           0.273642            0.334004            0.446680
Saved: /kaggle/working/legalrag_100k/bm25_vs_dense.json


# Reciprocal Rank Fusion (RRF)

## 20 Prepare BM25 and Dense results

In [30]:
if not (BASE_DIR / 'bm25_top50.json').exists():
    raise FileNotFoundError(
        'bm25_top50.json was not created. Run the BM25 retrieval stage first.'
    )

if not (BASE_DIR / 'dense_top50.json').exists():
    raise FileNotFoundError(
        'dense_top50.json was not created. Run dense retrieval first.'
    )

bm25_records = json.loads(
    (BASE_DIR / 'bm25_top50.json').read_text()
)

all_top50 = {
    str(cnr): pd.DataFrame(records)
    for cnr, records in bm25_records.items()
}

dense_top50 = json.loads(
    (BASE_DIR / 'dense_top50.json').read_text()
)

print('BM25 queries:', len(all_top50))
print('Dense queries:', len(dense_top50))


FileNotFoundError: bm25_top50.json was not created. Run the BM25 retrieval stage first.

## new one

In [31]:
# ============================================================
# Align saved retrieval-result filenames with notebook variables
# ============================================================

from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path("/kaggle/working/legalrag_100k")

BM25_RESULTS_PATH = BASE_DIR / "bm25_results.json"
DENSE_RESULTS_PATH = BASE_DIR / "dense_top50.json"

print("=" * 70)
print("RETRIEVAL CHECKPOINT ALIGNMENT")
print("=" * 70)

# ------------------------------------------------------------
# BM25
# ------------------------------------------------------------

if not BM25_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"BM25 results not found:\n{BM25_RESULTS_PATH}"
    )

bm25_records = json.loads(
    BM25_RESULTS_PATH.read_text()
)

all_top50 = {
    str(cnr): pd.DataFrame(records)
    for cnr, records in bm25_records.items()
}

print("BM25 results loaded:", len(all_top50))

# ------------------------------------------------------------
# Dense
# ------------------------------------------------------------

if not DENSE_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Dense results not found:\n{DENSE_RESULTS_PATH}"
    )

dense_top50 = json.loads(
    DENSE_RESULTS_PATH.read_text()
)

print("Dense results loaded:", len(dense_top50))

# ------------------------------------------------------------
# Alignment check
# ------------------------------------------------------------

expected_cnrs = {
    str(item["cnr"])
    for item in final_eval
}

bm25_cnrs = set(all_top50.keys())
dense_cnrs = set(str(cnr) for cnr in dense_top50.keys())

print()
print("Expected queries:", len(expected_cnrs))
print("BM25 queries:", len(bm25_cnrs))
print("Dense queries:", len(dense_cnrs))

if expected_cnrs != bm25_cnrs:
    raise ValueError(
        "BM25 result CNRs do not match final_eval."
    )

if expected_cnrs != dense_cnrs:
    raise ValueError(
        "Dense result CNRs do not match final_eval."
    )

print()
print("BM25 alignment ✓")
print("Dense alignment ✓")
print("=" * 70)

RETRIEVAL CHECKPOINT ALIGNMENT
BM25 results loaded: 497
Dense results loaded: 497

Expected queries: 497
BM25 queries: 497
Dense queries: 497

BM25 alignment ✓
Dense alignment ✓


## 21 Hybrid RRF

In [32]:
hybrid_top50 = {}

for item in final_eval:

    cnr = str(item['cnr'])
    bm25_results = all_top50[cnr].to_dict('records')
    dense_results = dense_top50[cnr]

    rrf_scores = {}
    chunk_info = {}

    for rank, row in enumerate(
        bm25_results,
        start=1
    ):
        chunk_id = str(row['chunk_id'])

        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0.0)
            + 1.0 / (RRF_K + rank)
        )

        chunk_info[chunk_id] = {
            'chunk_index': int(row['chunk_index']),
            'chunk_id': chunk_id,
            'cnr': str(row['cnr']),
        }

    for result in dense_results:
        rank = int(result['rank'])
        chunk_id = str(result['chunk_id'])

        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0.0)
            + 1.0 / (RRF_K + rank)
        )

        if chunk_id not in chunk_info:
            chunk_info[chunk_id] = {
                'chunk_index': int(result['chunk_index']),
                'chunk_id': chunk_id,
                'cnr': str(result['cnr']),
            }

    ranked = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    hybrid_top50[cnr] = [
        {
            'rank': rank,
            'chunk_index': chunk_info[chunk_id]['chunk_index'],
            'chunk_id': chunk_id,
            'cnr': chunk_info[chunk_id]['cnr'],
            'rrf_score': float(score),
        }
        for rank, (chunk_id, score) in enumerate(
            ranked[:RETRIEVAL_TOP_K],
            start=1
        )
    ]

    if len(hybrid_top50) % 25 == 0 or len(hybrid_top50) == len(final_eval):
        print(
            f'Processed {len(hybrid_top50)}/{len(final_eval)} | Hybrid RRF top-{RETRIEVAL_TOP_K}'
        )

print('Hybrid RRF retrieval complete ✓')


Processed 25/497 | Hybrid RRF top-50
Processed 50/497 | Hybrid RRF top-50
Processed 75/497 | Hybrid RRF top-50
Processed 100/497 | Hybrid RRF top-50
Processed 125/497 | Hybrid RRF top-50
Processed 150/497 | Hybrid RRF top-50
Processed 175/497 | Hybrid RRF top-50
Processed 200/497 | Hybrid RRF top-50
Processed 225/497 | Hybrid RRF top-50
Processed 250/497 | Hybrid RRF top-50
Processed 275/497 | Hybrid RRF top-50
Processed 300/497 | Hybrid RRF top-50
Processed 325/497 | Hybrid RRF top-50
Processed 350/497 | Hybrid RRF top-50
Processed 375/497 | Hybrid RRF top-50
Processed 400/497 | Hybrid RRF top-50
Processed 425/497 | Hybrid RRF top-50
Processed 450/497 | Hybrid RRF top-50
Processed 475/497 | Hybrid RRF top-50
Processed 497/497 | Hybrid RRF top-50
Hybrid RRF retrieval complete ✓


## 22 Hybrid Recall@K

In [33]:
hybrid_metrics = retrieval_metrics(
    hybrid_top50,
    'Hybrid-RRF'
)
hybrid_metrics['fusion'] = 'Reciprocal Rank Fusion'
hybrid_metrics['rrf_k'] = RRF_K

HYBRID_RESULTS_PATH = BASE_DIR / 'hybrid_top50.json'
HYBRID_METRICS_PATH = BASE_DIR / 'hybrid_metrics.json'

HYBRID_RESULTS_PATH.write_text(
    json.dumps(hybrid_top50, indent=2)
)
HYBRID_METRICS_PATH.write_text(
    json.dumps(hybrid_metrics, indent=2)
)

print('Saved:', HYBRID_RESULTS_PATH)
print('Saved:', HYBRID_METRICS_PATH)


Hybrid-RRF @ 1: chunk=0.1569 | judgment=0.3219
Hybrid-RRF @ 3: chunk=0.2636 | judgment=0.4527
Hybrid-RRF @ 5: chunk=0.3179 | judgment=0.5111
Hybrid-RRF @ 10: chunk=0.3763 | judgment=0.5614
Hybrid-RRF @ 25: chunk=0.4366 | judgment=0.6177
Hybrid-RRF @ 50: chunk=0.4688 | judgment=0.6579
Saved: /kaggle/working/legalrag_100k/hybrid_top50.json
Saved: /kaggle/working/legalrag_100k/hybrid_metrics.json


## 23 Cross-Encoder Reranker

In [34]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    RERANKER_MODEL,
    max_length=512,
    device='cuda:0'
)

print('Reranker:', RERANKER_MODEL)
print('Device: cuda:0')
print('Candidate depth:', RERANK_CANDIDATES)
print('Reranker loaded ✓')


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Device: cuda:0
Candidate depth: 50
Reranker loaded ✓


In [35]:
reranked_top50 = {}

for i, item in enumerate(final_eval, start=1):

    cnr = str(item['cnr'])
    question = str(item['question'])
    candidates = hybrid_top50[cnr][:RERANK_CANDIDATES]

    pairs = []

    for candidate in candidates:
        chunk_index = int(candidate['chunk_index'])
        chunk_text = str(
            chunks_df.iloc[chunk_index]['text']
        )
        pairs.append([question, chunk_text])

    scores = reranker.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    scores = np.asarray(
        scores,
        dtype=np.float32
    ).reshape(-1)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: float(x[1]),
        reverse=True
    )

    reranked_top50[cnr] = [
        {
            'rank': rank,
            'chunk_index': int(candidate['chunk_index']),
            'chunk_id': str(candidate['chunk_id']),
            'cnr': str(candidate['cnr']),
            'reranker_score': float(score),
        }
        for rank, (candidate, score) in enumerate(
            ranked[:RERANK_CANDIDATES],
            start=1
        )
    ]

    if i % 25 == 0 or i == len(final_eval):
        print(
            f'Processed {i}/{len(final_eval)} | Reranked {RERANK_CANDIDATES}'
        )

RERANKED_RESULTS_PATH = BASE_DIR / 'reranked_top50.json'
RERANKED_RESULTS_PATH.write_text(
    json.dumps(reranked_top50, indent=2)
)

print('Reranking complete ✓')
print('Saved:', RERANKED_RESULTS_PATH)


Processed 25/497 | Reranked 50
Processed 50/497 | Reranked 50
Processed 75/497 | Reranked 50
Processed 100/497 | Reranked 50
Processed 125/497 | Reranked 50
Processed 150/497 | Reranked 50
Processed 175/497 | Reranked 50
Processed 200/497 | Reranked 50
Processed 225/497 | Reranked 50
Processed 250/497 | Reranked 50
Processed 275/497 | Reranked 50
Processed 300/497 | Reranked 50
Processed 325/497 | Reranked 50
Processed 350/497 | Reranked 50
Processed 375/497 | Reranked 50
Processed 400/497 | Reranked 50
Processed 425/497 | Reranked 50
Processed 450/497 | Reranked 50
Processed 475/497 | Reranked 50
Processed 497/497 | Reranked 50
Reranking complete ✓
Saved: /kaggle/working/legalrag_100k/reranked_top50.json


## 24 Reranker Recall@K

In [36]:
reranker_metrics = retrieval_metrics(
    reranked_top50,
    'Hybrid-RRF + Cross-Encoder'
)
reranker_metrics['reranker_model'] = RERANKER_MODEL
reranker_metrics['reranker_candidates'] = RERANK_CANDIDATES

RERANKER_METRICS_PATH = BASE_DIR / 'reranker_metrics.json'
RERANKER_METRICS_PATH.write_text(
    json.dumps(reranker_metrics, indent=2)
)

print('Saved:', RERANKER_METRICS_PATH)


Hybrid-RRF + Cross-Encoder @ 1: chunk=0.1811 | judgment=0.3561
Hybrid-RRF + Cross-Encoder @ 3: chunk=0.2696 | judgment=0.4447
Hybrid-RRF + Cross-Encoder @ 5: chunk=0.2938 | judgment=0.4869
Hybrid-RRF + Cross-Encoder @ 10: chunk=0.3481 | judgment=0.5594
Hybrid-RRF + Cross-Encoder @ 25: chunk=0.4266 | judgment=0.6177
Hybrid-RRF + Cross-Encoder @ 50: chunk=0.4688 | judgment=0.6579
Saved: /kaggle/working/legalrag_100k/reranker_metrics.json


## 25 Final retrieval scoreboard

In [37]:
bm25_metrics = json.loads(
    (BASE_DIR / 'bm25_metrics.json').read_text()
)
dense_metrics = json.loads(
    (BASE_DIR / 'dense_metrics.json').read_text()
)
hybrid_metrics = json.loads(
    (BASE_DIR / 'hybrid_metrics.json').read_text()
)
reranker_metrics = json.loads(
    (BASE_DIR / 'reranker_metrics.json').read_text()
)

scoreboard = pd.DataFrame({
    'Retriever': [
        'BM25',
        'Dense',
        'Hybrid-RRF',
        'Hybrid-RRF + Reranker'
    ],
    'Judgment Recall@1': [
        bm25_metrics['judgment_recall@1'],
        dense_metrics['judgment_recall@1'],
        hybrid_metrics['judgment_recall@1'],
        reranker_metrics['judgment_recall@1']
    ],
    'Judgment Recall@5': [
        bm25_metrics['judgment_recall@5'],
        dense_metrics['judgment_recall@5'],
        hybrid_metrics['judgment_recall@5'],
        reranker_metrics['judgment_recall@5']
    ],
    'Judgment Recall@10': [
        bm25_metrics['judgment_recall@10'],
        dense_metrics['judgment_recall@10'],
        hybrid_metrics['judgment_recall@10'],
        reranker_metrics['judgment_recall@10']
    ],
    'Judgment Recall@50': [
        bm25_metrics['judgment_recall@50'],
        dense_metrics['judgment_recall@50'],
        hybrid_metrics['judgment_recall@50'],
        reranker_metrics['judgment_recall@50']
    ],
})

print(scoreboard.to_string(index=False))

SCOREBOARD_PATH = BASE_DIR / 'retrieval_scoreboard.json'
SCOREBOARD_PATH.write_text(
    scoreboard.to_json(
        orient='records',
        indent=2
    )
)

print('Saved:', SCOREBOARD_PATH)


            Retriever  Judgment Recall@1  Judgment Recall@5  Judgment Recall@10  Judgment Recall@50
                 BM25           0.426559           0.533199            0.577465            0.639839
                Dense           0.197183           0.273642            0.334004            0.446680
           Hybrid-RRF           0.321932           0.511066            0.561368            0.657948
Hybrid-RRF + Reranker           0.356137           0.486922            0.559356            0.657948
Saved: /kaggle/working/legalrag_100k/retrieval_scoreboard.json


# Checkpoint / backup

In [38]:
print('Project directory:', BASE_DIR)
print('\nTop-level artifacts:')

for path in sorted(BASE_DIR.iterdir()):
    if path.is_file():
        print(
            f'  {path.name} '
            f'({path.stat().st_size / (1024**2):.1f} MB)'
        )

embedding_shards = sorted(
    EMBED_DIR.glob('emb_*.npy')
)

print('\nEmbedding shards:', len(embedding_shards))

required = [
    CHUNKS_PATH,
    GOLD_EVAL_PATH,
    FAISS_PATH,
    DENSE_METRICS_PATH,
    HYBRID_METRICS_PATH,
    RERANKER_METRICS_PATH,
    SCOREBOARD_PATH,
]

missing = [
    str(path)
    for path in required
    if not path.exists()
]

if missing:
    print('Missing artifacts:')
    for path in missing:
        print(' ', path)
else:
    print('All retrieval-stage artifacts present ✓')


Project directory: /kaggle/working/legalrag_100k

Top-level artifacts:
  MANIFEST.json (0.0 MB)
  bm25.pkl (552.0 MB)
  bm25_metrics.json (0.0 MB)
  bm25_results.json (4.7 MB)
  bm25_vs_dense.json (0.0 MB)
  collection_manifest.json (0.0 MB)
  dense.index (1576.4 MB)
  dense_index_metadata.json (0.0 MB)
  dense_metrics.json (0.0 MB)
  dense_top50.json (4.6 MB)
  eval_candidates.json (0.5 MB)
  eval_pilot_10.json (0.0 MB)
  evaluation_documents.parquet (2.8 MB)
  experiment_state.json (0.0 MB)
  gold_eval.json (0.5 MB)
  hybrid_metrics.json (0.0 MB)
  hybrid_top50.json (4.7 MB)
  judgments_000.parquet (14.7 MB)
  judgments_001.parquet (13.3 MB)
  judgments_002.parquet (12.9 MB)
  judgments_003.parquet (13.1 MB)
  judgments_004.parquet (12.6 MB)
  judgments_005.parquet (12.4 MB)
  judgments_006.parquet (12.1 MB)
  judgments_007.parquet (11.7 MB)
  judgments_008.parquet (11.2 MB)
  judgments_009.parquet (11.4 MB)
  judgments_010.parquet (11.0 MB)
  judgments_011.parquet (11.5 MB)
  judgme

## save

In [47]:
# ============================================================
# LegalRAG 100k — Save Complete Progress Checkpoint
# ============================================================

from pathlib import Path
import shutil
import json

BASE_DIR = Path("/kaggle/working/legalrag_100k")
BACKUP_ROOT = Path("/kaggle/working/legalrag_100k_retrieval_checkpoint")

print("=" * 80)
print("SAVING LEGALRAG 100K RETRIEVAL CHECKPOINT")
print("=" * 80)

if not BASE_DIR.exists():
    raise FileNotFoundError(
        f"Workspace not found:\n{BASE_DIR}"
    )

# ------------------------------------------------------------
# Create clean checkpoint directory
# ------------------------------------------------------------

if BACKUP_ROOT.exists():
    shutil.rmtree(BACKUP_ROOT)

BACKUP_ROOT.mkdir(parents=True)

# ------------------------------------------------------------
# Copy complete LegalRAG workspace
# ------------------------------------------------------------

checkpoint_dir = BACKUP_ROOT / "legalrag_100k"

shutil.copytree(
    BASE_DIR,
    checkpoint_dir
)

print("Workspace copied ✓")

# ------------------------------------------------------------
# Create checkpoint metadata
# ------------------------------------------------------------

checkpoint_state = {
    "project": "LegalRAG",
    "experiment": "100k final experiment",

    "dataset_size": 100000,

    "validated_benchmark_questions": 497,
    "generated_benchmark_candidates": 500,

    "completed_stages": [
        "100k corpus",
        "chunking",
        "benchmark generation",
        "gold evidence validation",
        "BM25",
        "Dense retrieval",
        "Hybrid RRF",
        "Cross-encoder reranking"
    ],

    "current_next_stage": "RAG generation",

    "retrieval_configuration": {
        "candidate_pool": 50,
        "hybrid": "RRF",
        "reranker": "cross-encoder/ms-marco-MiniLM-L-6-v2"
    },

    "retrieval_metrics": {
        "BM25": {
            "recall_at_1": 0.426559,
            "recall_at_5": 0.533199,
            "recall_at_10": 0.577465,
            "recall_at_50": 0.639839
        },

        "Dense": {
            "recall_at_1": 0.197183,
            "recall_at_5": 0.273642,
            "recall_at_10": 0.334004,
            "recall_at_50": 0.446680
        },

        "Hybrid-RRF": {
            "recall_at_1": 0.321932,
            "recall_at_5": 0.511066,
            "recall_at_10": 0.561368,
            "recall_at_50": 0.657948
        },

        "Hybrid-RRF-Reranker": {
            "recall_at_1": 0.356137,
            "recall_at_5": 0.486922,
            "recall_at_10": 0.559356,
            "recall_at_50": 0.657948
        }
    }
}

state_path = (
    checkpoint_dir /
    "retrieval_checkpoint_state.json"
)

state_path.write_text(
    json.dumps(
        checkpoint_state,
        indent=2,
        ensure_ascii=False
    )
)

print("Checkpoint metadata saved ✓")

# ------------------------------------------------------------
# Create manifest
# ------------------------------------------------------------

manifest = {
    "project": "LegalRAG",
    "checkpoint": "100k retrieval complete",
    "files": []
}

for path in sorted(checkpoint_dir.rglob("*")):
    if path.is_file():
        manifest["files"].append({
            "path": str(
                path.relative_to(checkpoint_dir)
            ),
            "size_bytes": path.stat().st_size
        })

manifest_path = (
    checkpoint_dir /
    "CHECKPOINT_MANIFEST.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    )
)

# ------------------------------------------------------------
# Create downloadable ZIP
# ------------------------------------------------------------

zip_base = (
    Path("/kaggle/working")
    / "LegalRAG_100k_retrieval_checkpoint"
)

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=BACKUP_ROOT
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

zip_size_gb = (
    Path(zip_path).stat().st_size
    / (1024 ** 3)
)

print()
print("=" * 80)
print("CHECKPOINT COMPLETE ✓")
print("=" * 80)

print("Checkpoint folder:")
print(checkpoint_dir)

print()
print("ZIP:")
print(zip_path)

print(f"ZIP size: {zip_size_gb:.2f} GB")

print()
print("Next stage: RAG generation")
print("=" * 80)

SAVING LEGALRAG 100K RETRIEVAL CHECKPOINT
Workspace copied ✓
Checkpoint metadata saved ✓

CHECKPOINT COMPLETE ✓
Checkpoint folder:
/kaggle/working/legalrag_100k_retrieval_checkpoint/legalrag_100k

ZIP:
/kaggle/working/LegalRAG_100k_retrieval_checkpoint.zip
ZIP size: 3.84 GB

Next stage: RAG generation


## restart

In [5]:
# ============================================================
# Restore COMPLETE LegalRAG 100k checkpoint
# Input folder → Working folder
# ============================================================

from pathlib import Path
import shutil

SOURCE_DIR = Path(
    "/kaggle/input/datasets/"
    "siddharthdongardive/"
    "legalrag-dense-checkpoint/"
    "kaggle_100k_backup_before_rag_generation/"
    "legalrag_100k"
)

WORKING_DIR = Path(
    "/kaggle/working/legalrag_100k"
)

print("=" * 80)
print("RESTORING COMPLETE LEGALRAG 100K CHECKPOINT")
print("=" * 80)

if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"Input checkpoint does not exist:\n{SOURCE_DIR}"
    )

# Remove any old/incomplete working copy
if WORKING_DIR.exists():
    print("Removing existing working folder...")
    shutil.rmtree(WORKING_DIR)

# Copy EVERYTHING recursively
print("Copying complete checkpoint...")
shutil.copytree(
    SOURCE_DIR,
    WORKING_DIR
)

print()
print("Source:")
print(SOURCE_DIR)

print()
print("Working:")
print(WORKING_DIR)

# ------------------------------------------------------------
# Verify that everything was copied
# ------------------------------------------------------------

source_files = {
    path.relative_to(SOURCE_DIR)
    for path in SOURCE_DIR.rglob("*")
    if path.is_file()
}

working_files = {
    path.relative_to(WORKING_DIR)
    for path in WORKING_DIR.rglob("*")
    if path.is_file()
}

missing = source_files - working_files
extra = working_files - source_files

print()
print("=" * 80)
print("RESTORE VERIFICATION")
print("=" * 80)

print("Source files :", len(source_files))
print("Working files:", len(working_files))

if missing:
    print("\nMissing files:")
    for path in sorted(missing):
        print(" ", path)
else:
    print("Missing files: 0 ✓")

if extra:
    print("\nExtra files:")
    for path in sorted(extra):
        print(" ", path)
else:
    print("Extra files: 0 ✓")

if missing or extra:
    raise RuntimeError(
        "Checkpoint copy verification failed."
    )

print()
print("COMPLETE CHECKPOINT RESTORED ✓")
print("=" * 80)

RESTORING COMPLETE LEGALRAG 100K CHECKPOINT
Removing existing working folder...
Copying complete checkpoint...

Source:
/kaggle/input/datasets/siddharthdongardive/legalrag-dense-checkpoint/kaggle_100k_backup_before_rag_generation/legalrag_100k

Working:
/kaggle/working/legalrag_100k

RESTORE VERIFICATION
Source files : 154
Working files: 154
Missing files: 0 ✓
Extra files: 0 ✓

COMPLETE CHECKPOINT RESTORED ✓


In [41]:
# ============================================================
# LegalRAG 100k — Restore checkpoint variables into memory
# ============================================================

from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path("/kaggle/working/legalrag_100k")

# ------------------------------------------------------------
# Load chunk corpus
# ------------------------------------------------------------

CHUNKS_PATH = BASE_DIR / "legal_chunks.parquet"

chunks_df = pd.read_parquet(
    CHUNKS_PATH
).reset_index(drop=True)

# ------------------------------------------------------------
# Load final validated benchmark
# ------------------------------------------------------------

GOLD_EVAL_PATH = BASE_DIR / "gold_eval.json"

final_eval = json.loads(
    GOLD_EVAL_PATH.read_text()
)

# ------------------------------------------------------------
# Load reranked retrieval results
# ------------------------------------------------------------

RERANKED_PATH = BASE_DIR / "reranked_top50.json"

reranked_top50 = json.loads(
    RERANKED_PATH.read_text()
)

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print("=" * 70)
print("LEGALRAG 100K — CHECKPOINT VARIABLES RESTORED")
print("=" * 70)

print("chunks_df:", len(chunks_df))
print("final_eval:", len(final_eval))
print("reranked_top50:", len(reranked_top50))

if len(final_eval) != 497:
    raise ValueError(
        f"Expected 497 final evaluation records, "
        f"found {len(final_eval)}"
    )

if len(reranked_top50) != 497:
    raise ValueError(
        f"Expected 497 reranked query results, "
        f"found {len(reranked_top50)}"
    )

expected_cnrs = {
    str(item["cnr"])
    for item in final_eval
}

retrieved_cnrs = {
    str(cnr)
    for cnr in reranked_top50.keys()
}

missing = expected_cnrs - retrieved_cnrs

if missing:
    raise ValueError(
        f"{len(missing)} CNRs are missing from reranked_top50."
    )

print("✓ chunks_df restored")
print("✓ final_eval restored")
print("✓ reranked_top50 restored")
print("✓ 497/497 CNRs aligned")
print()
print("READY FOR YOUR EXISTING RAG CONTEXT CELL")
print("=" * 70)

LEGALRAG 100K — CHECKPOINT VARIABLES RESTORED
chunks_df: 538079
final_eval: 497
reranked_top50: 497
✓ chunks_df restored
✓ final_eval restored
✓ reranked_top50 restored
✓ 497/497 CNRs aligned

READY FOR YOUR EXISTING RAG CONTEXT CELL


# Phase 3 — RAG Generation

The retrieval pipeline above is the experimental core. The next stage generates grounded answers from the reranked legal context.

All model generation uses the same OmniRoute Combo **`new`**.


## 26 RAG context builder

In [9]:
CONTEXT_TOP_K = 5
rag_contexts = {}

for item in final_eval:

    cnr = str(item['cnr'])
    context_chunks = []

    for result in reranked_top50[cnr][:CONTEXT_TOP_K]:

        chunk_index = int(result['chunk_index'])
        row = chunks_df.iloc[chunk_index]

        context_chunks.append({
            'rank': int(result['rank']),
            'chunk_id': str(row['chunk_id']),
            'cnr': str(row['cnr']),
            'court_code': str(row['court_code']),
            'decision_date': str(row['decision_date']),
            'case_type': str(row['case_type']),
            'title': str(row['title']),
            'text': str(row['text']),
            'reranker_score': float(result['reranker_score']),
        })

    rag_contexts[cnr] = {
        'question': str(item['question']),
        'chunks': context_chunks,
    }

print('RAG contexts:', len(rag_contexts))
print('Context chunks per question:', CONTEXT_TOP_K)


RAG contexts: 497
Context chunks per question: 5


## 27 Single-question RAG test

In [10]:
test_cnr = next(iter(rag_contexts))
test_item = rag_contexts[test_cnr]

context_text = "\n\n".join(
    [
        f"[Rank {chunk['rank']}]\n"
        f"Chunk ID: {chunk['chunk_id']}\n"
        f"Court: {chunk['court_code']}\n"
        f"Date: {chunk['decision_date']}\n"
        f"Case: {chunk['title']}\n\n"
        f"{chunk['text']}"
        for chunk in test_item['chunks']
    ]
)

RAG_PROMPT_TEMPLATE = '''
You are a legal RAG assistant.

Answer the user's question using ONLY the retrieved judgment excerpts below.
Do not use outside knowledge.
Do not invent facts, legal provisions, reasoning, or conclusions.

Give a concise but complete answer.
Where a claim is supported by a retrieved excerpt, cite its exact Chunk ID.
Do not cite a chunk that does not support the claim.

USER QUESTION:
{question}

RETRIEVED JUDGMENT EXCERPTS:
{context}

ANSWER:
'''

RAG_PROMPT = RAG_PROMPT_TEMPLATE.format(
    question=test_item['question'],
    context=context_text
)

response = client.chat.completions.create(
    model="kaggle",
    messages=[{'role': 'user', 'content': RAG_PROMPT}],
    temperature=0
)

test_answer = response.choices[0].message.content.strip()

print('Requested model:', "kaggle")
print('CNR:', test_cnr)
print('\nQUESTION:')
print(test_item['question'])
print('\nGENERATED ANSWER:')
print(test_answer)


Requested model: kaggle
CNR: BRHC011179452023_1_2025-11-11

QUESTION:
What was the High Court's final decision regarding the appellant's prayer for anticipatory bail?

GENERATED ANSWER:
Based on the retrieved excerpt for the specific case you asked about (CR. APP (SJ)/5177/2024 of Lila Devi Vs The State of Bihar), the High Court rejected the appelant prayer for anticipatory bail.

The excerpt shows that the Special Court had already rejected the prayer for anticipatory bail, and the appeal was heard before the High Court. While the chunk includes the submissions made by both parties (including that the informant does not belong to SC/ST category and that the informant's caste "Tanti" has been removed from the list of Scheduled Castes), it does not explicitly state the High Court's final order. However, given that the case was an appeal against the rejection of anticipatory bail and no subsequent order allowing the bail is mentioned, the prayer remains rejected.


## 28 Generate RAG answers for the full evaluation set

In [17]:
RAG_RESULTS_PATH = BASE_DIR / 'rag_results.json'

if RAG_RESULTS_PATH.exists():
    rag_results = json.loads(RAG_RESULTS_PATH.read_text())
    completed_cnrs = {str(item['cnr']) for item in rag_results}
    print('Loaded existing results:', len(rag_results))
else:
    rag_results = []
    completed_cnrs = set()

for i, (cnr, item) in enumerate(rag_contexts.items(), start=1):

    cnr = str(cnr)

    if cnr in completed_cnrs:
        print(f'[{i}/{len(rag_contexts)}] SKIP | {cnr}')
        continue

    context_text = "\n\n".join(
        [
            f"[Rank {chunk['rank']}]\n"
            f"Chunk ID: {chunk['chunk_id']}\n"
            f"Court: {chunk['court_code']}\n"
            f"Date: {chunk['decision_date']}\n"
            f"Case: {chunk['title']}\n\n"
            f"{chunk['text']}"
            for chunk in item['chunks']
        ]
    )

    prompt = RAG_PROMPT_TEMPLATE.format(
        question=item['question'],
        context=context_text
    )

    print('\n' + '-' * 70)
    print(f'[{i}/{len(rag_contexts)}] GENERATING')
    print('CNR:', cnr)

    try:
        response = client.chat.completions.create(
            model=OMNIROUTE_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0
        )

        answer = response.choices[0].message.content.strip()

        if not answer:
            raise ValueError('OmniRoute returned an empty answer.')

        result = {
            'cnr': cnr,
            'question': item['question'],
            'answer': answer,
            'context_chunks': item['chunks'],
        }

        rag_results.append(result)
        completed_cnrs.add(cnr)

        RAG_RESULTS_PATH.write_text(
            json.dumps(
                rag_results,
                indent=2,
                ensure_ascii=False
            )
        )

        print(f'SUCCESS | Progress: {len(rag_results)}/{EVAL_DOCUMENTS}')
        print('Checkpoint saved:', RAG_RESULTS_PATH)

    except Exception as e:
        print('\nRAG GENERATION ERROR')
        print('CNR:', cnr)
        print('Error:', repr(e))
        print('Completed answers remain saved.')
        raise

print('\nRAG answers completed:', len(rag_results))

if len(rag_results) != EVAL_DOCUMENTS:
    raise RuntimeError(
        f'Expected {EVAL_DOCUMENTS} RAG answers, found {len(rag_results)}.'
    )


Loaded existing results: 497
[1/497] SKIP | BRHC011179452023_1_2025-11-11
[2/497] SKIP | SKHC010001152022_1_2023-08-01
[3/497] SKIP | MNHC010011422022_1_2023-08-02
[4/497] SKIP | TRHC010005342024_1_2025-03-18
[5/497] SKIP | MLHC010000022025_1_2025-06-12
[6/497] SKIP | GAHC010086642020_1_2020-07-17
[7/497] SKIP | WBCHCA0182542020_1_2020-09-03
[8/497] SKIP | JKHC010058272025_1_2025-11-08
[9/497] SKIP | JHHC010431392023_1_2024-01-29
[10/497] SKIP | ODHC010074862000_1_2016-03-01
[11/497] SKIP | CGHC010002962020_1_2020-01-16
[12/497] SKIP | MPHC030130242020_1_2020-08-14
[13/497] SKIP | GJHC240026661996_1_1996-03-15
[14/497] SKIP | HCBM030440212017_1_2019-01-10
[15/497] SKIP | KAHC010237931999_1_1999-12-15
[16/497] SKIP | HPHC010014761992_1_2006-11-07
[17/497] SKIP | KLHC010211752008_1_2008-01-28
[18/497] SKIP | HCMA011084922016_1_2016-07-11
[19/497] SKIP | HBHC010074822008_1_2008-07-18
[20/497] SKIP | PHHC010653812008_1_2008-12-23
[21/497] SKIP | UKHC010009192001_1_2001-11-03
[22/497] SKIP 

RuntimeError: Expected 500 RAG answers, found 497.

In [16]:
# ============================================================
# LegalRAG 100k — Restore checkpoint variables into memory
# ============================================================

from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path("/kaggle/working/legalrag_100k")

# ------------------------------------------------------------
# Load chunk corpus
# ------------------------------------------------------------

CHUNKS_PATH = BASE_DIR / "legal_chunks.parquet"

chunks_df = pd.read_parquet(
    CHUNKS_PATH
).reset_index(drop=True)

# ------------------------------------------------------------
# Load final validated benchmark
# ------------------------------------------------------------

GOLD_EVAL_PATH = BASE_DIR / "gold_eval.json"

final_eval = json.loads(
    GOLD_EVAL_PATH.read_text()
)

# ------------------------------------------------------------
# Load reranked retrieval results
# ------------------------------------------------------------

RERANKED_PATH = BASE_DIR / "reranked_top50.json"

reranked_top50 = json.loads(
    RERANKED_PATH.read_text()
)

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print("=" * 70)
print("LEGALRAG 100K — CHECKPOINT VARIABLES RESTORED")
print("=" * 70)

print("chunks_df:", len(chunks_df))
print("final_eval:", len(final_eval))
print("reranked_top50:", len(reranked_top50))

if len(final_eval) != 497:
    raise ValueError(
        f"Expected 497 final evaluation records, "
        f"found {len(final_eval)}"
    )

if len(reranked_top50) != 497:
    raise ValueError(
        f"Expected 497 reranked query results, "
        f"found {len(reranked_top50)}"
    )

expected_cnrs = {
    str(item["cnr"])
    for item in final_eval
}

retrieved_cnrs = {
    str(cnr)
    for cnr in reranked_top50.keys()
}

missing = expected_cnrs - retrieved_cnrs

if missing:
    raise ValueError(
        f"{len(missing)} CNRs are missing from reranked_top50."
    )

print("✓ chunks_df restored")
print("✓ final_eval restored")
print("✓ reranked_top50 restored")
print("✓ 497/497 CNRs aligned")
print()
print("READY FOR YOUR EXISTING RAG CONTEXT CELL")
print("=" * 70)

LEGALRAG 100K — CHECKPOINT VARIABLES RESTORED
chunks_df: 538079
final_eval: 497
reranked_top50: 497
✓ chunks_df restored
✓ final_eval restored
✓ reranked_top50 restored
✓ 497/497 CNRs aligned

READY FOR YOUR EXISTING RAG CONTEXT CELL


In [18]:
CONTEXT_TOP_K = 5
rag_contexts = {}

for item in final_eval:

    cnr = str(item['cnr'])
    context_chunks = []

    for result in reranked_top50[cnr][:CONTEXT_TOP_K]:

        chunk_index = int(result['chunk_index'])
        row = chunks_df.iloc[chunk_index]

        context_chunks.append({
            'rank': int(result['rank']),
            'chunk_id': str(row['chunk_id']),
            'cnr': str(row['cnr']),
            'court_code': str(row['court_code']),
            'decision_date': str(row['decision_date']),
            'case_type': str(row['case_type']),
            'title': str(row['title']),
            'text': str(row['text']),
            'reranker_score': float(result['reranker_score']),
        })

    rag_contexts[cnr] = {
        'question': str(item['question']),
        'chunks': context_chunks,
    }

print('RAG contexts:', len(rag_contexts))
print('Context chunks per question:', CONTEXT_TOP_K)

RAG contexts: 497
Context chunks per question: 5


## 29 Prepare RAG evaluation records

In [20]:
rag_results = json.loads(
    (BASE_DIR / 'rag_results.json').read_text()
)

gold_eval = json.loads(
    GOLD_EVAL_PATH.read_text()
)

gold_by_cnr = {
    str(item['cnr']): item
    for item in gold_eval
}

evaluation_records = []

for result in rag_results:
    cnr = str(result['cnr'])
    gold = gold_by_cnr.get(cnr)

    if gold is None:
        raise ValueError(
            f'No gold benchmark item found for CNR {cnr}.'
        )

    evaluation_records.append({
        'cnr': cnr,
        'question': result['question'],
        'generated_answer': result['answer'],
        'reference_answer': gold['reference_answer'],
        'question_type': gold['question_type'],
        'supporting_text': gold['supporting_text'],
        'context_chunks': result['context_chunks'],
    })

print('Evaluation records:', len(evaluation_records))

FINAL_EVAL_DOCUMENTS = len(gold_eval)

if len(evaluation_records) != FINAL_EVAL_DOCUMENTS:
    raise ValueError(
        f'Expected {FINAL_EVAL_DOCUMENTS} evaluation records, '
        f'found {len(evaluation_records)}.'
    )

print('Evaluation records validated ✓')

Evaluation records: 497
Evaluation records validated ✓


## 30 RAG evaluation judge

In [23]:
RAG_EVAL_PATH = BASE_DIR / 'rag_evaluation.json'

RAG_JUDGE_PROMPT_TEMPLATE = '''
You are evaluating the answer produced by a legal RAG system.

Evaluate the GENERATED ANSWER using ONLY:
1. The USER QUESTION
2. The REFERENCE ANSWER
3. The SUPPORTING TEXT
4. The RETRIEVED CONTEXT

Score the answer on these four dimensions:

1. answer_relevance
   - 1 = does not answer the question
   - 2 = partially relevant
   - 3 = mostly relevant
   - 4 = directly and adequately answers the question

2. faithfulness
   - 1 = substantial unsupported or contradictory claims
   - 2 = several unsupported claims
   - 3 = mostly supported, minor unsupported details
   - 4 = fully supported by the retrieved context

3. citation_correctness
   - 1 = citations are absent when needed or clearly incorrect
   - 2 = some citation support but incomplete/incorrect
   - 3 = mostly correct citations
   - 4 = citations correctly point to supporting retrieved chunks
   - If there are no citations but the answer is otherwise grounded, score 2.

4. unsupported_claim_rate
   - Return a number from 0.0 to 1.0.
   - 0.0 means no meaningful unsupported claims.
   - 1.0 means essentially all substantive claims are unsupported.

Also provide:
- overall_score: 1 to 4
- hallucination: true or false
- explanation: concise reason for the scores

Return ONLY valid JSON in exactly this structure:
{
  "answer_relevance": 1,
  "faithfulness": 1,
  "citation_correctness": 1,
  "unsupported_claim_rate": 0.0,
  "overall_score": 1,
  "hallucination": false,
  "explanation": "..."
}

USER QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

SUPPORTING TEXT:
{supporting_text}

RETRIEVED CONTEXT:
{retrieved_context}

GENERATED ANSWER:
{generated_answer}
'''

def parse_judge_response(text):
    text = text.strip()

    # Handle plain JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Handle ```json ... ``` responses
    if text.startswith("```"):
        lines = text.splitlines()

        if lines[0].startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        cleaned = "\n".join(lines).strip()

        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            pass

    # Try extracting the first JSON object from surrounding text
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]

        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass

    raise ValueError(
        f"Could not parse judge response as JSON:\n{text}"
    )

# Validate one judge call before starting the 500-record run.
test_record = evaluation_records[0]

test_context = "\n\n".join(
    [
        f"[Chunk {chunk['chunk_id']}]\n{chunk['text']}"
        for chunk in test_record['context_chunks']
    ]
)

test_prompt = (
    RAG_JUDGE_PROMPT_TEMPLATE
    .replace("{question}", test_record["question"])
    .replace("{reference_answer}", test_record["reference_answer"])
    .replace("{supporting_text}", test_record["supporting_text"])
    .replace("{retrieved_context}", test_context)
    .replace("{generated_answer}", test_record["generated_answer"])
)

response = client.chat.completions.create(
    model=OMNIROUTE_MODEL,
    messages=[{'role': 'user', 'content': test_prompt}],
    temperature=0
)

judge_result = parse_judge_response(
    response.choices[0].message.content
)

required_judge_fields = {
    'answer_relevance',
    'faithfulness',
    'citation_correctness',
    'unsupported_claim_rate',
    'overall_score',
    'hallucination',
    'explanation',
}

if not required_judge_fields.issubset(judge_result):
    raise ValueError(
        f'Judge response missing fields: {required_judge_fields - set(judge_result)}'
    )

print('=' * 70)
print('RAG EVALUATION JUDGE TEST')
print('=' * 70)
print('Requested model:', OMNIROUTE_MODEL)
print('Progress: 1 /', len(evaluation_records))
print('CNR:', test_record['cnr'])
print('Judge call successful ✓')
print(json.dumps(judge_result, indent=2, ensure_ascii=False))


RAG EVALUATION JUDGE TEST
Requested model: kaggle
Progress: 1 / 497
CNR: BRHC011179452023_1_2025-11-11
Judge call successful ✓
{
  "answer_relevance": 2,
  "faithfulness": 2,
  "citation_correctness": 2,
  "unsupported_claim_rate": 0.3,
  "overall_score": 2,
  "hallucination": false,
  "explanation": "The generated answer discusses multiple different cases with varying outcomes (some granting anticipatory bail, some rejecting it), which creates confusion and doesn't directly answer the specific question about 'the appellant' in the reference answer. While the reference answer clearly states the High Court rejected the prayer and dismissed the appeal, the generated answer mixes information from various cases without providing a clear, focused response to the specific query. The citations point to retrieved chunks but are not correctly targeted to answer the question asked."
}


## 31 Run RAG evaluation with checkpointing

In [44]:
if RAG_EVAL_PATH.exists():
    rag_evaluations = json.loads(
        RAG_EVAL_PATH.read_text()
    )
    completed_cnrs = {
        str(item['cnr'])
        for item in rag_evaluations
    }
    print('Loaded existing evaluations:', len(rag_evaluations))
else:
    rag_evaluations = []
    completed_cnrs = set()
    print('No existing evaluation checkpoint found. Starting fresh.')

print('=' * 70)
print('STARTING RAG EVALUATION')
print('Total records:', len(evaluation_records))
print('Already done:', len(completed_cnrs))
print('Remaining:', len(evaluation_records) - len(completed_cnrs))
print('=' * 70)

for idx, record in enumerate(evaluation_records, start=1):

    cnr = str(record['cnr'])

    if cnr in completed_cnrs:
        print(f'[{idx}/{len(evaluation_records)}] SKIP | {cnr}')
        continue

    retrieved_context = "\n\n".join(
        [
            f"[Chunk {chunk['chunk_id']}]\n{chunk['text']}"
            for chunk in record['context_chunks']
        ]
    )

    prompt = (
    RAG_JUDGE_PROMPT_TEMPLATE
    .replace("{question}", record["question"])
    .replace("{reference_answer}", record["reference_answer"])
    .replace("{supporting_text}", record["supporting_text"])
    .replace("{retrieved_context}", retrieved_context)
    .replace("{generated_answer}", record["generated_answer"])
)

    print('\n' + '-' * 70)
    print(f'[{idx}/{len(evaluation_records)}] EVALUATING')
    print('CNR:', cnr)

    try:
        response = client.chat.completions.create(
            model=OMNIROUTE_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0
        )

        result = parse_judge_response(
            response.choices[0].message.content
        )

        if not required_judge_fields.issubset(result):
            raise ValueError(
                f'Judge response missing fields: {required_judge_fields - set(result)}'
            )

        evaluation = {
            'cnr': cnr,
            'question': record['question'],
            'answer_relevance': int(result['answer_relevance']),
            'faithfulness': int(result['faithfulness']),
            'citation_correctness': int(result['citation_correctness']),
            'unsupported_claim_rate': float(result['unsupported_claim_rate']),
            'overall_score': int(result['overall_score']),
            'hallucination': bool(result['hallucination']),
            'explanation': str(result['explanation']),
        }

        rag_evaluations.append(evaluation)
        completed_cnrs.add(cnr)

        RAG_EVAL_PATH.write_text(
            json.dumps(
                rag_evaluations,
                indent=2,
                ensure_ascii=False
            )
        )

        print(f'SUCCESS | Progress: {len(rag_evaluations)}/{len(evaluation_records)}')
        print(
            f"Scores → relevance={evaluation['answer_relevance']} | "
            f"faithfulness={evaluation['faithfulness']} | "
            f"citation={evaluation['citation_correctness']} | "
            f"overall={evaluation['overall_score']} | "
            f"hallucination={evaluation['hallucination']}"
        )
        print('Checkpoint saved:', RAG_EVAL_PATH)

    except Exception as e:
        print('\nRAG EVALUATION ERROR')
        print('CNR:', cnr)
        print('Error:', repr(e))
        print('Completed evaluations remain saved.')
        raise

print('\n' + '=' * 70)
print('RAG EVALUATION RUN FINISHED')
print(f'Completed: {len(rag_evaluations)}/{len(evaluation_records)}')
print('Checkpoint:', RAG_EVAL_PATH)
print('=' * 70)

if len(rag_evaluations) != len(evaluation_records):
    raise RuntimeError(
        f'Expected {len(evaluation_records)} evaluations, found {len(rag_evaluations)}.'
    )


Loaded existing evaluations: 109
STARTING RAG EVALUATION
Total records: 497
Already done: 109
Remaining: 388
[1/497] SKIP | BRHC011179452023_1_2025-11-11
[2/497] SKIP | SKHC010001152022_1_2023-08-01
[3/497] SKIP | MNHC010011422022_1_2023-08-02
[4/497] SKIP | TRHC010005342024_1_2025-03-18
[5/497] SKIP | MLHC010000022025_1_2025-06-12
[6/497] SKIP | GAHC010086642020_1_2020-07-17
[7/497] SKIP | WBCHCA0182542020_1_2020-09-03
[8/497] SKIP | JKHC010058272025_1_2025-11-08
[9/497] SKIP | JHHC010431392023_1_2024-01-29
[10/497] SKIP | ODHC010074862000_1_2016-03-01
[11/497] SKIP | CGHC010002962020_1_2020-01-16
[12/497] SKIP | MPHC030130242020_1_2020-08-14
[13/497] SKIP | GJHC240026661996_1_1996-03-15
[14/497] SKIP | HCBM030440212017_1_2019-01-10
[15/497] SKIP | KAHC010237931999_1_1999-12-15
[16/497] SKIP | HPHC010014761992_1_2006-11-07
[17/497] SKIP | KLHC010211752008_1_2008-01-28
[18/497] SKIP | HCMA011084922016_1_2016-07-11
[19/497] SKIP | HBHC010074822008_1_2008-07-18
[20/497] SKIP | PHHC010653

## 32 RAG evaluation summary

In [45]:
rag_evaluations = json.loads(
    RAG_EVAL_PATH.read_text()
)

eval_df = pd.DataFrame(rag_evaluations)

print('=' * 70)
print('RAG EVALUATION SUMMARY')
print('=' * 70)
print('Total evaluated records:', len(eval_df))
print()

aggregate_metrics = {
    'total_records': int(len(eval_df)),
    'answer_relevance': float(eval_df['answer_relevance'].mean()),
    'faithfulness': float(eval_df['faithfulness'].mean()),
    'citation_correctness': float(eval_df['citation_correctness'].mean()),
    'unsupported_claim_rate': float(eval_df['unsupported_claim_rate'].mean()),
    'hallucination_rate': float(eval_df['hallucination'].mean()),
    'overall_score': float(eval_df['overall_score'].mean()),
}

for key, value in aggregate_metrics.items():
    if key == 'total_records':
        continue
    suffix = ' / 4' if key not in {'unsupported_claim_rate', 'hallucination_rate'} else ''
    print(f'{key.replace("_", " ").title():30s}: {value:.4f}{suffix}')

print()
print('Score distributions:')
for column in [
    'answer_relevance',
    'faithfulness',
    'citation_correctness',
    'overall_score'
]:
    print(f'{column}:')
    print(
        eval_df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

print()
print('Hallucination distribution:')
print(eval_df['hallucination'].value_counts().to_dict())

RAG_METRICS_PATH = BASE_DIR / 'rag_metrics.json'
RAG_METRICS_PATH.write_text(
    json.dumps(aggregate_metrics, indent=2)
)

eval_df.to_csv(
    BASE_DIR / 'rag_evaluation.csv',
    index=False
)

print('\nSaved:', RAG_METRICS_PATH)


RAG EVALUATION SUMMARY
Total evaluated records: 497

Answer Relevance              : 3.0966 / 4
Faithfulness                  : 3.4004 / 4
Citation Correctness          : 3.2515 / 4
Unsupported Claim Rate        : 0.1489
Hallucination Rate            : 0.2274
Overall Score                 : 3.0926 / 4

Score distributions:
answer_relevance:
{1: 88, 2: 69, 3: 47, 4: 293}
faithfulness:
{1: 53, 2: 53, 3: 33, 4: 358}
citation_correctness:
{1: 46, 2: 100, 3: 34, 4: 317}
overall_score:
{1: 67, 2: 90, 3: 70, 4: 270}

Hallucination distribution:
{False: 384, True: 113}

Saved: /kaggle/working/legalrag_100k/rag_metrics.json


## 33 RAG evaluation by question type

In [46]:
question_type_map = {
    str(record['cnr']): record.get('question_type')
    for record in evaluation_records
}

eval_df['question_type'] = eval_df['cnr'].map(question_type_map)

summary_by_type = (
    eval_df
    .groupby('question_type')
    .agg(
        records=('cnr', 'count'),
        answer_relevance=('answer_relevance', 'mean'),
        faithfulness=('faithfulness', 'mean'),
        citation_correctness=('citation_correctness', 'mean'),
        unsupported_claim_rate=('unsupported_claim_rate', 'mean'),
        hallucination_rate=('hallucination', 'mean'),
        overall_score=('overall_score', 'mean'),
    )
    .sort_values('overall_score', ascending=False)
)

print('=' * 70)
print('RAG EVALUATION BY QUESTION TYPE')
print('=' * 70)
print(summary_by_type.round(4).to_string())

print('\nHallucination counts by question type:')
print(
    pd.crosstab(
        eval_df['question_type'],
        eval_df['hallucination']
    )
)

summary_by_type.to_csv(
    BASE_DIR / 'rag_evaluation_by_question_type.csv'
)


RAG EVALUATION BY QUESTION TYPE
                 records  answer_relevance  faithfulness  citation_correctness  unsupported_claim_rate  hallucination_rate  overall_score
question_type                                                                                                                            
legal_provision       80            3.5000        3.6625                3.6375                  0.0906              0.1500         3.4375
fact                 100            3.3300        3.4400                3.2700                  0.1388              0.2300         3.1400
reasoning            149            3.0604        3.5235                3.4027                  0.1250              0.1745         3.1007
multi_hop             50            2.7600        3.4000                3.2600                  0.1306              0.2400         3.0200
outcome              118            2.8136        3.0339                2.7797                  0.2347              0.3390         2.8390

H

## 34 Citation → judgment consistency

In [47]:
import re

citation_rows = []

for _, row in eval_df.iterrows():

    cnr = str(row['cnr'])
    record = next(
        r for r in evaluation_records
        if str(r['cnr']) == cnr
    )

    answer = str(record['generated_answer'])

    cited_chunk_ids = re.findall(
        r'\b[A-Z]{2,}HC[A-Z0-9_]+_\d{4}-\d{2}-\d{2}_\d+\b',
        answer
    )

    retrieved_chunk_ids = {
        str(chunk['chunk_id']): str(chunk['cnr'])
        for chunk in record['context_chunks']
    }

    correct_citations = 0
    wrong_citations = 0
    not_retrieved = 0

    for chunk_id in cited_chunk_ids:
        if chunk_id not in retrieved_chunk_ids:
            not_retrieved += 1
        elif retrieved_chunk_ids[chunk_id] != cnr:
            wrong_citations += 1
        else:
            correct_citations += 1

    citation_rows.append({
        'cnr': cnr,
        'question_type': row['question_type'],
        'citations_found': len(cited_chunk_ids),
        'correct_cnr_citations': correct_citations,
        'wrong_cnr_citations': wrong_citations,
        'citations_not_retrieved': not_retrieved,
        'hallucination': row['hallucination'],
        'unsupported_claim_rate': row['unsupported_claim_rate'],
    })

citation_df = pd.DataFrame(citation_rows)

print('=' * 70)
print('CITATION → JUDGMENT CONSISTENCY')
print('=' * 70)
print(
    f"Answers with citations found        : {(citation_df['citations_found'] > 0).sum()}"
)
print(
    f"Answers citing correct judgment     : {(citation_df['correct_cnr_citations'] > 0).sum()}"
)
print(
    f"Answers citing wrong judgment       : {(citation_df['wrong_cnr_citations'] > 0).sum()}"
)
print(
    f"Answers citing non-retrieved chunks : {(citation_df['citations_not_retrieved'] > 0).sum()}"
)

print('\nBy question type:')
print(
    citation_df.groupby('question_type').agg(
        records=('cnr', 'count'),
        with_citations=('citations_found', lambda x: (x > 0).sum()),
        correct_cnr_citation=('correct_cnr_citations', lambda x: (x > 0).sum()),
        wrong_cnr_citation=('wrong_cnr_citations', lambda x: (x > 0).sum()),
        non_retrieved_citation=('citations_not_retrieved', lambda x: (x > 0).sum()),
    ).to_string()
)

citation_df.to_csv(
    BASE_DIR / 'citation_analysis.csv',
    index=False
)


CITATION → JUDGMENT CONSISTENCY
Answers with citations found        : 399
Answers citing correct judgment     : 191
Answers citing wrong judgment       : 233
Answers citing non-retrieved chunks : 10

By question type:
                 records  with_citations  correct_cnr_citation  wrong_cnr_citation  non_retrieved_citation
question_type                                                                                             
fact                 100              77                    44                  37                       2
legal_provision       80              72                    27                  54                       1
multi_hop             50              42                    23                  22                       2
outcome              118              89                    50                  44                       3
reasoning            149             119                    47                  76                       2


## 35 Correct judgment presence in RAG context

In [48]:
context_rows = []

for _, row in eval_df.iterrows():

    cnr = str(row['cnr'])
    record = next(
        r for r in evaluation_records
        if str(r['cnr']) == cnr
    )

    correct_chunks = [
        chunk for chunk in record['context_chunks']
        if str(chunk['cnr']) == cnr
    ]

    context_rows.append({
        'cnr': cnr,
        'question_type': row['question_type'],
        'context_size': len(record['context_chunks']),
        'correct_judgment_chunks': len(correct_chunks),
        'correct_judgment_present': len(correct_chunks) > 0,
        'hallucination': row['hallucination'],
        'unsupported_claim_rate': row['unsupported_claim_rate'],
        'answer_relevance': row['answer_relevance'],
    })

context_df = pd.DataFrame(context_rows)

print('=' * 70)
print('CORRECT JUDGMENT PRESENCE IN RAG CONTEXT')
print('=' * 70)
print(
    f"Questions with correct judgment in context: "
    f"{context_df['correct_judgment_present'].sum()} / {len(context_df)}"
)
print(
    f"Questions with NO correct judgment in context: "
    f"{(~context_df['correct_judgment_present']).sum()} / {len(context_df)}"
)

print('\nDistribution of correct-judgment chunks:')
print(
    context_df['correct_judgment_chunks']
    .value_counts()
    .sort_index()
    .to_string()
)

print('\nBy question type:')
print(
    context_df.groupby('question_type').agg(
        records=('cnr', 'count'),
        correct_present=('correct_judgment_present', 'sum'),
        correct_absent=('correct_judgment_present', lambda x: (~x).sum()),
        avg_correct_chunks=('correct_judgment_chunks', 'mean'),
        hallucination_rate=('hallucination', 'mean'),
        avg_unsupported_claim_rate=('unsupported_claim_rate', 'mean'),
    ).round(4).to_string()
)

context_df.to_csv(
    BASE_DIR / 'context_analysis.csv',
    index=False
)


CORRECT JUDGMENT PRESENCE IN RAG CONTEXT
Questions with correct judgment in context: 242 / 497
Questions with NO correct judgment in context: 255 / 497

Distribution of correct-judgment chunks:
correct_judgment_chunks
0    255
1    166
2     44
3     16
4      8
5      8

By question type:
                 records  correct_present  correct_absent  avg_correct_chunks  hallucination_rate  avg_unsupported_claim_rate
question_type                                                                                                                
fact                 100               54              46              0.8700              0.2300                      0.1388
legal_provision       80               36              44              0.6125              0.1500                      0.0906
multi_hop             50               25              25              0.8200              0.2400                      0.1306
outcome              118               64              54              0.7458  

## 36 Retrieval vs generation failure analysis

In [49]:
# Classify failure modes without treating every unrelated context chunk as a retrieval failure.

failure_rows = []
ERROR_MARKERS = [
    'no longer available',
    'switch to',
]

for _, row in eval_df.iterrows():

    cnr = str(row['cnr'])
    record = next(
        r for r in evaluation_records
        if str(r['cnr']) == cnr
    )

    answer = str(record['generated_answer']).lower()

    correct_present = any(
        str(chunk['cnr']) == cnr
        for chunk in record['context_chunks']
    )

    has_model_error = any(
        marker in answer
        for marker in ERROR_MARKERS
    )

    if has_model_error:
        failure_type = 'generation_or_api_error'
    elif not correct_present:
        failure_type = 'retrieval_failure'
    elif row['hallucination'] and row['citation_correctness'] <= 2:
        failure_type = 'context_selection_or_citation_failure'
    elif row['hallucination']:
        failure_type = 'generation_grounding_failure'
    elif row['answer_relevance'] <= 2:
        failure_type = 'low_answer_relevance'
    else:
        failure_type = 'no_major_failure'

    failure_rows.append({
        'cnr': cnr,
        'question_type': row['question_type'],
        'failure_type': failure_type,
        'answer_relevance': row['answer_relevance'],
        'faithfulness': row['faithfulness'],
        'citation_correctness': row['citation_correctness'],
        'unsupported_claim_rate': row['unsupported_claim_rate'],
        'hallucination': row['hallucination'],
    })

failure_df = pd.DataFrame(failure_rows)

print('=' * 70)
print('RAG FAILURE ANALYSIS')
print('=' * 70)
print('\nFailure type distribution:')
print(
    failure_df['failure_type']
    .value_counts()
    .to_string()
)

print('\nFailure type by question type:')
print(
    pd.crosstab(
        failure_df['question_type'],
        failure_df['failure_type']
    )
)

failure_df.to_csv(
    BASE_DIR / 'failure_analysis.csv',
    index=False
)


RAG FAILURE ANALYSIS

Failure type distribution:
failure_type
retrieval_failure                        255
no_major_failure                         194
low_answer_relevance                      21
context_selection_or_citation_failure     19
generation_grounding_failure               8

Failure type by question type:
failure_type     context_selection_or_citation_failure  \
question_type                                            
fact                                                 1   
legal_provision                                      0   
multi_hop                                            3   
outcome                                             11   
reasoning                                            4   

failure_type     generation_grounding_failure  low_answer_relevance  \
question_type                                                         
fact                                        4                     2   
legal_provision                             0                

## 37 Final experiment checkpoint

In [50]:
ANALYSIS_DIR = BASE_DIR / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# Save the core analysis tables.
for filename in [
    'rag_evaluation.csv',
    'citation_analysis.csv',
    'context_analysis.csv',
    'failure_analysis.csv',
    'rag_evaluation_by_question_type.csv',
]:
    source = BASE_DIR / filename
    if source.exists():
        shutil.copy2(source, ANALYSIS_DIR / filename)

for filename in [
    'rag_metrics.json',
    'bm25_metrics.json',
    'dense_metrics.json',
    'hybrid_metrics.json',
    'reranker_metrics.json',
    'retrieval_scoreboard.json',
]:
    source = BASE_DIR / filename
    if source.exists():
        shutil.copy2(source, ANALYSIS_DIR / filename)

FINAL_MANIFEST = {
    'project': 'LegalRAG',
    'dataset': DATASET_NAME,
    'dataset_config': DATASET_CONFIG,
    'dataset_split': DATASET_SPLIT,
    'corpus_judgments': TARGET_JUDGMENTS,
    'evaluation_questions': EVAL_DOCUMENTS,
    'pilot_questions': PILOT_DOCUMENTS,
    'chunk_size': CHUNK_SIZE,
    'chunk_overlap': CHUNK_OVERLAP,
    'min_chunk_length': MIN_CHUNK_LENGTH,
    'embedding_model': MODEL_NAME,
    'embedding_dimension': EMBED_DIM,
    'retrieval_top_k': RETRIEVAL_TOP_K,
    'rrf_k': RRF_K,
    'reranker_model': RERANKER_MODEL,
    'reranker_candidates': RERANK_CANDIDATES,
    'context_top_k': CONTEXT_TOP_K,
    'omniroute_model': OMNIROUTE_MODEL,
    'omniroute_base_url': OMNIROUTE_BASE_URL,
    'question_type_targets': QUESTION_TYPE_TARGETS,
}

(BASE_DIR / 'final_experiment_manifest.json').write_text(
    json.dumps(FINAL_MANIFEST, indent=2)
)

print('=' * 70)
print('FINAL EXPERIMENT CHECKPOINT')
print('=' * 70)
print('Corpus:', f'{TARGET_JUDGMENTS:,} judgments')
print('Benchmark:', f'{EVAL_DOCUMENTS:,} questions')
print('RAG answers:', f'{len(rag_results):,}')
print('RAG evaluations:', f'{len(rag_evaluations):,}')
print('Workspace:', BASE_DIR)
print('Analysis:', ANALYSIS_DIR)
print('All final experiment artifacts saved ✓')


FINAL EXPERIMENT CHECKPOINT
Corpus: 100,000 judgments
Benchmark: 500 questions
RAG answers: 497
RAG evaluations: 497
Workspace: /kaggle/working/legalrag_100k
Analysis: /kaggle/working/legalrag_100k/analysis
All final experiment artifacts saved ✓


## 38 Package final experiment

In [51]:
ARCHIVE_PATH = Path('/kaggle/working/LegalRAG_100k_final_experiment')

archive_file = Path(
    shutil.make_archive(
        str(ARCHIVE_PATH),
        'zip',
        BASE_DIR
    )
)

print('Archive created:')
print(archive_file)
print(
    f'Archive size: {archive_file.stat().st_size / (1024**3):.2f} GB'
)


Archive created:
/kaggle/working/LegalRAG_100k_final_experiment.zip
Archive size: 3.84 GB
